# Arabic document VLM, part 1b (v6): data for the roadmap

**Runtime: CPU is enough** (no GPU needed; one optional cell in section 7 is faster on a GPU). Run this notebook
once, then again whenever you add one of the optional inputs below. Every step skips work that is already saved, so
re-running only does what is new. Everything goes into `MyDrive/arabic_vlm/data_v6`, which the v6 training notebook
reads.

**What this notebook builds, by roadmap phase**

| Phase | What | Section |
| --- | --- | --- |
| 0 | the **frozen test suite**: every test set, with its text-disjoint ("headline") subset fixed once in `eval_suite.json` | 4 |
| 0 | the **4-gram and image near-copy filter**: every training source is checked against every test set | 5 |
| 0 | the **writer audit** of our Teklia KHATT copy (needs the official KHATT release) | 6 |
| 0, 5 | the **held-out form set**: blank templates you download, filled in code, printed and photographed, matched by QR code | 7 |
| 2 | KHATT with each fixed passage capped at 3 copies; the official unique-text lines with writer IDs; a seen-writer test | 3, 5 |
| 2, 3 | new handwriting: the Nakba memoir lines (Omar Al-Saleh collection) and AraMS-28k (14 manuscript books, 3 unseen in test) | 3 |
| 2 | more handwriting-style and calligraphic fonts for rendered handwriting lines | 1 |
| 4 | English: GNHK phone-photo handwriting lines, olmOCR-mix pages, 20,000 SynthDoG pages, IAM validation | 3 |
| 5 | names, places and field types for the form generator v2; filled training templates | 2, 7 |

**Optional inputs you can add to `MyDrive/arabic_vlm/`** (each section explains its own):

| Folder | What to put there | Used for |
| --- | --- | --- |
| `khatt_official/` | the official KHATT release from KFUPM ([request it here](http://khatt.ideas2serve.net/)), zipped or unzipped | unique-text lines with writer IDs, the unseen-writer and seen-writer tests, the writer audit |
| `form_templates/` | 40–60 **blank** PDF or image forms you downloaded from public sites (never filled ones) | the held-out form set and training templates |
| `form_photos/` | phone photos of the printed held-out forms | the held-out form set, real capture |
| `unlabelled_lines/` | line images without transcriptions | self-labelling in the training notebook |
| `real_docs/` | your own anonymised documents with `labels.csv` (as in v5) | evaluation only |

**Gated dataset.** The memoir lines (`U4RASD/omar-al-saleh-manuscripts-segments`, CC BY 4.0) are gated: open the dataset
page on Hugging Face while logged in, accept the conditions, and add your token as the Colab secret `HF_TOKEN`
(key icon in the left bar). Without it, that source is skipped and everything else still runs.

**Privacy.** Forms are built only from blank templates and fictional values made in code: random combinations of
common first and family names, made-up numbers in the usual formats, example e-mail domains. No real person's data,
no real ID, and no official identity-document design is used anywhere.

**Time and space (full size).** About 2–3 hours on a CPU runtime, mostly downloads (SynthDoG pages, AraMS-28k,
GNHK) and image checks. About 12–15 GB on Drive (section 10 can delete the unfiltered downloads afterwards). The v5
data folder is reused where possible, so nothing that v5 already downloaded is fetched again.

In [ ]:
%pip install -q datasets fonttools huggingface_hub pypdfium2 qrcode augraphy

In [ ]:
import os, sys, re, io, csv, glob, json, time, math, shutil, random, hashlib, tarfile, zipfile, unicodedata, subprocess, urllib.request
from collections import Counter, defaultdict
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageChops, ImageFilter, features

Image.MAX_IMAGE_PIXELS = None    # manuscript pages and phone photos can be large
QUICK_TEST = os.environ.get("ARVL_QUICK_TEST", "0") == "1"   # tiny run, used only to test this notebook
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    try:                         # a Hugging Face token (Colab secret "HF_TOKEN"): faster downloads, and the gated memoir lines
        from google.colab import userdata
        os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
    except Exception:
        print("No HF_TOKEN secret found: public downloads work; the gated memoir lines will be skipped")
HF_TOKEN = os.environ.get("HF_TOKEN") or None

# --- folders ------------------------------------------------------------------------------
BASE_DIR = os.environ.get("ARVL_BASE", "/content/drive/MyDrive/arabic_vlm" if IN_COLAB else "arabic_vlm")
DATA_DIR = os.environ.get("ARVL_DATA", os.path.join(BASE_DIR, "data_v6"))        # everything this notebook saves
V5_DATA_DIR = os.environ.get("ARVL_DATA_V5", os.path.join(BASE_DIR, "data_v5"))  # reused where possible
WORK_DIR = os.environ.get("ARVL_WORK", "/content/arvl_work" if IN_COLAB else DATA_DIR + "_work")   # local scratch (archives, shards), not on Drive
KHATT_OFFICIAL_DIR = os.path.join(BASE_DIR, "khatt_official")
FORM_TEMPLATES_DIR = os.path.join(BASE_DIR, "form_templates")
FORM_PHOTOS_DIR = os.path.join(BASE_DIR, "form_photos")
UNLABELLED_DIR = os.path.join(BASE_DIR, "unlabelled_lines")
REAL_DOCS_DIR = os.path.join(BASE_DIR, "real_docs")
IAM_OFFICIAL_DIR = os.path.join(BASE_DIR, "iam_official")   # optional: IAM's ascii/lines.txt and forms.txt, for the IAM writer audit
V5_CHECKPOINT = os.environ.get("ARVL_V5_STAGE3", os.path.join(BASE_DIR, "stage3_v5"))   # used only by the optional label-reading cell in section 7

# --- switches -----------------------------------------------------------------------------
USE_MEMOIRS = True               # Nakba memoir lines (gated: needs HF_TOKEN and the accepted conditions)
USE_ARAMS = True                 # AraMS-28k-HTR (2.4 GB zip; CC BY-NC-SA 4.0)
USE_GNHK = True                  # GNHK English handwriting in phone photos (CC BY 4.0)
USE_OLMOCR_EVAL = True           # olmOCR-mix-1025 evaluation pages (191 MB archive; ODC-BY)
USE_OLMOCR_TRAIN = False         # one olmOCR-mix training archive (about 1 GB) for printed English pages
USE_FUNSD = True                 # FUNSD's 50 test forms (real scanned English forms), evaluation only
USE_SYNTHDOG = True              # rendered English pages (SynthDoG-en)
N_SYNTHDOG = 20_000              # v5 used 2,000; the roadmap asks for 20,000-50,000 (about 150 KB each at the settings below)
SYNTHDOG_MAX_WORDS = 150         # v5: 80. Longer pages for phase 4 ("lines grow from 20 words to whole paragraphs")
SYNTHDOG_MAX_SIDE = 1024
OLMOCR_EVAL_PAGES = 400          # of the 2,288 evaluation pages (English, no tables or diagrams, at most OLMOCR_MAX_WORDS)
OLMOCR_MAX_WORDS = 400
SUITE_MAX = 500                  # images per test set in the suite (KITAB sets: the first 500, as in v5)
KHATT_REPEAT_CAP = 3             # keep at most this many copies of each repeated KHATT passage
SEEN_WRITERS = 100               # official KHATT only: training writers who each give one unique-text paragraph to the seen-writer test
REDO = False                     # False: anything already saved in DATA_DIR is kept
if QUICK_TEST:
    N_SYNTHDOG, OLMOCR_EVAL_PAGES, SUITE_MAX, SEEN_WRITERS = 6, 4, 8, 2
LIMIT = 12 if QUICK_TEST else None    # rows per downloaded split in a test run

for sub in ["fonts", "text", "datasets", "raw", "forms"]:
    os.makedirs(os.path.join(DATA_DIR, sub), exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
print("saving to", os.path.abspath(DATA_DIR), "| reusing v5 data from", V5_DATA_DIR if os.path.isdir(V5_DATA_DIR) else "(no v5 folder found)")
MANIFEST_PATH = os.path.join(DATA_DIR, "manifest.json")
manifest = json.load(open(MANIFEST_PATH)) if os.path.exists(MANIFEST_PATH) else {}
manifest.update({"version": "v6", "created": manifest.get("created", time.strftime("%Y-%m-%d %H:%M")),
                 "updated": time.strftime("%Y-%m-%d %H:%M")})
manifest.setdefault("parts", {})


def note(part, **info):
    """Record what a step produced (counts, terms) in the manifest, and save it, so a later crash keeps the record."""
    manifest["parts"].setdefault(part, {}).update(info)
    json.dump(manifest, open(MANIFEST_PATH, "w"), indent=1, ensure_ascii=False)

In [ ]:
from datasets import load_dataset, load_from_disk, Dataset, Features, Value, Sequence, concatenate_datasets, Image as ImageFeature


def dataset_path(name, folder="datasets"):
    return os.path.join(DATA_DIR, folder, name)


def saved_dataset(name, folder="datasets"):
    """The saved dataset `name` if this notebook already wrote it (and REDO is off), else None."""
    path = dataset_path(name, folder)
    if REDO or not os.path.exists(os.path.join(path, "state.json")):
        return None
    return load_from_disk(path)


def save(ds, name, folder="datasets"):
    """Save a dataset (atomically) and return the saved copy. An empty dataset is not saved: it counts as missing."""
    if len(ds) == 0:
        print(f"   {name}: empty, not saved")
        return ds
    path = dataset_path(name, folder)
    tmp = path + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    ds.save_to_disk(tmp)
    shutil.rmtree(path, ignore_errors=True)
    os.replace(tmp, path)                   # a half-written dataset never looks finished
    return load_from_disk(path)


def image_bytes(img, fmt="PNG", quality=90):
    """A PIL image as the {bytes, path} dict the datasets Image feature stores, so every image lives inside the dataset
    files (a path would break once the folder is copied to another machine)."""
    buf = io.BytesIO()
    if fmt == "JPEG":
        img.convert("RGB").save(buf, "JPEG", quality=quality)
    else:
        img.save(buf, fmt)
    return {"bytes": buf.getvalue(), "path": None}


class ShardWriter:
    """Builds a dataset 2,000 rows at a time on local disk, so decoded images never pile up in memory (v5's first
    version ran Colab out of RAM that way) and nothing depends on the datasets cache."""

    def __init__(self, name, features, rows_per_shard=2000):
        self.dir = os.path.join(WORK_DIR, "shards", name)
        shutil.rmtree(self.dir, ignore_errors=True)
        os.makedirs(self.dir)
        self.features, self.rows_per_shard, self.rows, self.shards, self.count = features, rows_per_shard, [], [], 0

    def add(self, row):
        self.rows.append(row)
        self.count += 1
        if len(self.rows) >= self.rows_per_shard:
            self.flush()

    def flush(self):
        if self.rows:
            path = os.path.join(self.dir, f"{len(self.shards):05d}")
            Dataset.from_list(self.rows, features=self.features).save_to_disk(path)
            self.shards.append(path)
            self.rows = []

    def finish(self):
        self.flush()
        if not self.shards:
            return Dataset.from_dict({k: [] for k in self.features}, features=self.features)
        return concatenate_datasets([load_from_disk(p) for p in self.shards])


LINE_FEATURES = Features({"image": ImageFeature(), "text": Value("string"), "source": Value("string"),
                          "writer": Value("string"), "group": Value("string"), "note": Value("string")})
# writer: a writer or form ID where the source has one ("" otherwise); group: a page, book or paragraph ID used to keep
# related lines on one side of a split; note: anything else worth keeping (line type, file name)


def line_row(img, text, source, writer="", group="", note="", fmt="PNG"):
    return {"image": image_bytes(img, fmt) if isinstance(img, Image.Image) else img, "text": text, "source": source,
            "writer": writer or "", "group": group or "", "note": note or ""}


# --- text comparison: one definition everywhere (the training notebook uses the same) -----
TASHKEEL = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۜ۟-۪ۨ-ۭـ]")
LETTER_MAP = str.maketrans({**{"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي", "ة": "ه", "ی": "ي", "ک": "ك"},
                            **dict(zip("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789"))})


def plain(text):
    """Text reduced for comparing: no diacritics or tatweel, alef/yeh/teh marbuta unified, Western digits, lower case,
    punctuation removed, single spaces."""
    text = TASHKEEL.sub("", unicodedata.normalize("NFC", text or "")).translate(LETTER_MAP).lower()
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", text)).strip()


def ngrams(text, n=4):
    words = plain(text).split()
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}


def fingerprint(img):
    """64-bit difference hash: near-identical images (rescaled, recompressed) get nearly the same bits."""
    small = np.asarray(img.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return int("".join("1" if b else "0" for b in (small[:, 1:] > small[:, :-1]).flatten()), 2)


def fingerprint_stored(stored):
    """Fingerprint of an image stored as {bytes, path} (a dataset column read with decode=False)."""
    with Image.open(io.BytesIO(stored["bytes"]) if stored.get("bytes") else stored["path"]) as img:
        return fingerprint(img)


_POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def near_copy_mask(prints, reference, max_bits=4):
    """For each fingerprint in `prints`, True if it is within `max_bits` bits of any fingerprint in `reference`."""
    out = np.zeros(len(prints), dtype=bool)
    if len(prints) == 0 or len(reference) == 0:
        return out
    a, b = np.array(prints, dtype=np.uint64), np.array(reference, dtype=np.uint64)
    for start in range(0, len(a), 1024):
        x = a[start:start + 1024, None] ^ b[None, :]
        bits = _POPCOUNT[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)
        out[start:start + 1024] = (bits <= max_bits).any(1)
    return out


def text_column(ds):
    for name in ["text", "answer", "label", "transcription", "ground_truth", "gt_text"]:
        if name in ds.column_names:
            return name
    raise KeyError(f"no text column in {ds.column_names}")


def first(ds, n):
    return ds if n is None else ds.select(range(min(n, len(ds))))


def copy_v5(name, folder="datasets", new_name=None):
    """Copy a dataset that part 1b v5 already saved, instead of downloading it again. Returns it, or None."""
    target = new_name or name
    ds = saved_dataset(target, folder)
    if ds is not None:
        return ds
    source = os.path.join(V5_DATA_DIR, "datasets", name)
    if not os.path.exists(os.path.join(source, "state.json")):
        return None
    shutil.rmtree(dataset_path(target, folder), ignore_errors=True)
    shutil.copytree(source, dataset_path(target, folder))
    print(f"   {target}: copied from the v5 data folder")
    return load_from_disk(dataset_path(target, folder))


V5_TRAINING = {"khatt_teklia_train": "khatt_train", "muharaf_train": "muharaf_train", "iam_train": "iam_train"}


def raw_dataset(name):
    """A training source as downloaded: from raw/ in this folder or, for the three v5 already has, read in place from
    the v5 folder (not copied, to save Drive space)."""
    ds = saved_dataset(name, "raw")
    if ds is not None:
        return ds
    v5 = os.path.join(V5_DATA_DIR, "datasets", V5_TRAINING.get(name, "-"))
    return load_from_disk(v5) if name in V5_TRAINING and os.path.exists(os.path.join(v5, "state.json")) else None


def hf_file(repo, filename, repo_type="dataset"):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo, filename, repo_type=repo_type, token=HF_TOKEN)


def hf_folder(repo, allow_patterns=None, repo_type="dataset"):
    from huggingface_hub import snapshot_download
    return snapshot_download(repo, repo_type=repo_type, allow_patterns=allow_patterns, token=HF_TOKEN,
                             local_dir=os.path.join(WORK_DIR, "hub", repo.replace("/", "__")))


def step(title):
    """Print a section banner and return a timer."""
    print(f"\n=== {title}")
    return time.time()

## 1. Fonts
The v5 fonts (copied from the v5 folder when it is there) plus new ones for training: two more Arabic hands for the
rendered handwriting lines of phase 2, **Gulzar** and **Noto Nastaliq Urdu** (Nastaliq calligraphy), two display
faces (**Reem Kufi Fun**, **Badeen Display**), 12 more English handwriting-like fonts and five more English text
fonts; and six new fonts kept for the form test only (below). Every font is checked for every glyph it must draw (Arabic fonts also for Latin
letters and digits, which they draw in mixed lines); a font missing any is skipped, so no image shows a missing-glyph
box. Colour fonts (Reem Kufi Ink, Blaka Ink, Cairo Play) are left out because Pillow draws them blank or flat.

**Fonts held out for the form test.** Six fonts are never used in training, so the held-out forms are typed and
"handwritten" in fonts no model has seen: Fustat (Arabic text), Alkalami (an Arabic handwriting style from West
Africa), Public Sans, Space Mono, Reenie Beanie and Handlee (English). `fonts.json` lists them under `heldout_forms`, and the training notebook removes them from every
training renderer.

In [ ]:
from fontTools.ttLib import TTFont

GOOGLE_FONTS = "https://raw.githubusercontent.com/google/fonts/main/"
ARABIC_FONT_FILES = {   # file name: path in the google/fonts repository
    # book and newspaper text
    "NotoNaskhArabic.ttf": "ofl/notonaskharabic/NotoNaskhArabic%5Bwght%5D.ttf",
    "NotoNaskhArabicUI.ttf": "ofl/notonaskharabicui/NotoNaskhArabicUI%5Bwght%5D.ttf",
    "Amiri.ttf": "ofl/amiri/Amiri-Regular.ttf", "Amiri-Bold.ttf": "ofl/amiri/Amiri-Bold.ttf",
    "AmiriQuran.ttf": "ofl/amiriquran/AmiriQuran-Regular.ttf",
    "ScheherazadeNew.ttf": "ofl/scheherazadenew/ScheherazadeNew-Regular.ttf",
    "Lateef.ttf": "ofl/lateef/Lateef-Regular.ttf", "Harmattan.ttf": "ofl/harmattan/Harmattan-Regular.ttf",
    "MarkaziText.ttf": "ofl/markazitext/MarkaziText%5Bwght%5D.ttf", "Mirza.ttf": "ofl/mirza/Mirza-Regular.ttf",
    "Ruwudu.ttf": "ofl/ruwudu/Ruwudu-Regular.ttf",
    # modern sans: forms, screens, slides, signs
    "NotoSansArabic.ttf": "ofl/notosansarabic/NotoSansArabic%5Bwdth,wght%5D.ttf",
    "NotoSansArabicUI.ttf": "ofl/notosansarabicui/NotoSansArabicUI%5Bwdth,wght%5D.ttf",
    "NotoKufiArabic.ttf": "ofl/notokufiarabic/NotoKufiArabic%5Bwght%5D.ttf",
    "Cairo.ttf": "ofl/cairo/Cairo%5Bslnt,wght%5D.ttf", "Tajawal.ttf": "ofl/tajawal/Tajawal-Regular.ttf",
    "Almarai.ttf": "ofl/almarai/Almarai-Regular.ttf", "IBMPlexSansArabic.ttf": "ofl/ibmplexsansarabic/IBMPlexSansArabic-Regular.ttf",
    "ElMessiri.ttf": "ofl/elmessiri/ElMessiri%5Bwght%5D.ttf", "Changa.ttf": "ofl/changa/Changa%5Bwght%5D.ttf",
    "ReemKufi.ttf": "ofl/reemkufi/ReemKufi%5Bwght%5D.ttf", "Mada.ttf": "ofl/mada/Mada%5Bwght%5D.ttf",
    "ReadexPro.ttf": "ofl/readexpro/ReadexPro%5BHEXP,wght%5D.ttf", "Alexandria.ttf": "ofl/alexandria/Alexandria%5Bwght%5D.ttf",
    "BalooBhaijaan2.ttf": "ofl/baloobhaijaan2/BalooBhaijaan2%5Bwght%5D.ttf", "Vazirmatn.ttf": "ofl/vazirmatn/Vazirmatn%5Bwght%5D.ttf",
    "Lemonada.ttf": "ofl/lemonada/Lemonada%5Bwght%5D.ttf",
    "Zain.ttf": "ofl/zain/Zain-Regular.ttf", "Beiruti.ttf": "ofl/beiruti/Beiruti%5Bwght%5D.ttf", "Rubik.ttf": "ofl/rubik/Rubik%5Bwght%5D.ttf",
    "Fustat.ttf": "ofl/fustat/Fustat%5Bwght%5D.ttf",                                   # new in v6 (held out for forms)
    # handwriting-like and calligraphic
    "ArefRuqaa.ttf": "ofl/arefruqaa/ArefRuqaa-Regular.ttf", "ArefRuqaaInk.ttf": "ofl/arefruqaaink/ArefRuqaaInk-Regular.ttf",
    "Marhey.ttf": "ofl/marhey/Marhey%5Bwght%5D.ttf", "Rakkas.ttf": "ofl/rakkas/Rakkas-Regular.ttf",
    "PlaypenSansArabic.ttf": "ofl/playpensansarabic/PlaypenSansArabic%5Bwght%5D.ttf",
    "Alkalami.ttf": "ofl/alkalami/Alkalami-Regular.ttf",                               # new in v6 (held out for forms)
    "Gulzar.ttf": "ofl/gulzar/Gulzar-Regular.ttf",                                     # new in v6
    "NotoNastaliqUrdu.ttf": "ofl/notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf",     # new in v6
    # heavy display (headlines, signs)
    "Lalezar.ttf": "ofl/lalezar/Lalezar-Regular.ttf", "Katibeh.ttf": "ofl/katibeh/Katibeh-Regular.ttf",
    "Jomhuria.ttf": "ofl/jomhuria/Jomhuria-Regular.ttf", "Blaka.ttf": "ofl/blaka/Blaka-Regular.ttf",
    "ReemKufiFun.ttf": "ofl/reemkufifun/ReemKufiFun%5Bwght%5D.ttf", "BadeenDisplay.ttf": "ofl/badeendisplay/BadeenDisplay-Regular.ttf",
}
ENGLISH_FONT_FILES = {
    # sans
    "NotoSans.ttf": "ofl/notosans/NotoSans%5Bwdth,wght%5D.ttf", "OpenSans.ttf": "ofl/opensans/OpenSans%5Bwdth,wght%5D.ttf",
    "Lato.ttf": "ofl/lato/Lato-Regular.ttf", "Roboto.ttf": "ofl/roboto/Roboto%5Bwdth,wght%5D.ttf",
    "Inter.ttf": "ofl/inter/Inter%5Bopsz,wght%5D.ttf", "WorkSans.ttf": "ofl/worksans/WorkSans%5Bwght%5D.ttf",
    "DMSans.ttf": "ofl/dmsans/DMSans%5Bopsz,wght%5D.ttf", "Poppins.ttf": "ofl/poppins/Poppins-Regular.ttf",
    "Montserrat.ttf": "ofl/montserrat/Montserrat%5Bwght%5D.ttf", "Raleway.ttf": "ofl/raleway/Raleway%5Bwght%5D.ttf",
    "Nunito.ttf": "ofl/nunito/Nunito%5Bwght%5D.ttf", "Oswald.ttf": "ofl/oswald/Oswald%5Bwght%5D.ttf",
    "Arimo.ttf": "ofl/arimo/Arimo%5Bwght%5D.ttf", "Cabin.ttf": "ofl/cabin/Cabin%5Bwdth,wght%5D.ttf",
    "Karla.ttf": "ofl/karla/Karla%5Bwght%5D.ttf", "Lexend.ttf": "ofl/lexend/Lexend%5Bwght%5D.ttf",
    "Barlow.ttf": "ofl/barlow/Barlow-Regular.ttf", "Rubik-Latin.ttf": "ofl/rubik/Rubik%5Bwght%5D.ttf",
    "SourceSans3.ttf": "ofl/sourcesans3/SourceSans3%5Bwght%5D.ttf", "Mulish.ttf": "ofl/mulish/Mulish%5Bwght%5D.ttf",   # new in v6
    "FiraSans.ttf": "ofl/firasans/FiraSans-Regular.ttf", "Archivo.ttf": "ofl/archivo/Archivo%5Bwdth,wght%5D.ttf",     # new in v6
    "IBMPlexSans.ttf": "ofl/ibmplexsans/IBMPlexSans%5Bwdth,wght%5D.ttf",                                              # new in v6
    "PublicSans.ttf": "ofl/publicsans/PublicSans%5Bwght%5D.ttf",                                                      # new in v6 (held out)
    # serif
    "NotoSerif.ttf": "ofl/notoserif/NotoSerif%5Bwdth,wght%5D.ttf", "Merriweather.ttf": "ofl/merriweather/Merriweather%5Bopsz,wdth,wght%5D.ttf",
    "SourceSerif4.ttf": "ofl/sourceserif4/SourceSerif4%5Bopsz,wght%5D.ttf", "PTSerif.ttf": "ofl/ptserif/PT_Serif-Web-Regular.ttf",
    "EBGaramond.ttf": "ofl/ebgaramond/EBGaramond%5Bwght%5D.ttf", "PlayfairDisplay.ttf": "ofl/playfairdisplay/PlayfairDisplay%5Bwght%5D.ttf",
    "Tinos.ttf": "ofl/tinos/Tinos-Regular.ttf", "CrimsonText.ttf": "ofl/crimsontext/CrimsonText-Regular.ttf", "Lora.ttf": "ofl/lora/Lora%5Bwght%5D.ttf",
    # monospace (codes, reference numbers, IDs)
    "CourierPrime.ttf": "ofl/courierprime/CourierPrime-Regular.ttf", "IBMPlexMono.ttf": "ofl/ibmplexmono/IBMPlexMono-Regular.ttf",
    "RobotoMono.ttf": "ofl/robotomono/RobotoMono%5Bwght%5D.ttf", "SourceCodePro.ttf": "ofl/sourcecodepro/SourceCodePro%5Bwght%5D.ttf",
    "Cousine.ttf": "ofl/cousine/Cousine-Regular.ttf", "JetBrainsMono.ttf": "ofl/jetbrainsmono/JetBrainsMono%5Bwght%5D.ttf",
    "SpaceMono.ttf": "ofl/spacemono/SpaceMono-Regular.ttf",                                                           # new in v6 (held out)
    # handwriting-like
    "Caveat.ttf": "ofl/caveat/Caveat%5Bwght%5D.ttf", "PatrickHand.ttf": "ofl/patrickhand/PatrickHand-Regular.ttf",
    "DancingScript.ttf": "ofl/dancingscript/DancingScript%5Bwght%5D.ttf", "IndieFlower.ttf": "ofl/indieflower/IndieFlower-Regular.ttf",
    "ShadowsIntoLight.ttf": "ofl/shadowsintolight/ShadowsIntoLight.ttf", "Kalam.ttf": "ofl/kalam/Kalam-Regular.ttf",
    "ArchitectsDaughter.ttf": "ofl/architectsdaughter/ArchitectsDaughter-Regular.ttf", "GloriaHallelujah.ttf": "ofl/gloriahallelujah/GloriaHallelujah.ttf",
    # handwriting-like, new in v6
    "NothingYouCouldDo.ttf": "ofl/nothingyoucoulddo/NothingYouCouldDo.ttf", "CoveredByYourGrace.ttf": "ofl/coveredbyyourgrace/CoveredByYourGrace.ttf",
    "GochiHand.ttf": "ofl/gochihand/GochiHand-Regular.ttf", "Neucha.ttf": "ofl/neucha/Neucha.ttf", "Pangolin.ttf": "ofl/pangolin/Pangolin-Regular.ttf",
    "ShortStack.ttf": "ofl/shortstack/ShortStack-Regular.ttf", "SueEllenFrancisco.ttf": "ofl/sueellenfrancisco/SueEllenFrancisco-Regular.ttf",
    "NanumPenScript.ttf": "ofl/nanumpenscript/NanumPenScript-Regular.ttf", "Schoolbell.ttf": "apache/schoolbell/Schoolbell-Regular.ttf",
    "ComingSoon.ttf": "apache/comingsoon/ComingSoon-Regular.ttf", "RockSalt.ttf": "apache/rocksalt/RockSalt-Regular.ttf",
    "HomemadeApple.ttf": "apache/homemadeapple/HomemadeApple-Regular.ttf",
    "ReenieBeanie.ttf": "ofl/reeniebeanie/ReenieBeanie.ttf", "Handlee.ttf": "ofl/handlee/Handlee-Regular.ttf",       # held out
}
FONT_STYLE = {   # used by the training notebook to choose fonts per kind of image
    "handwriting_like": ["ArefRuqaa.ttf", "ArefRuqaaInk.ttf", "Marhey.ttf", "Rakkas.ttf", "PlaypenSansArabic.ttf", "Ruwudu.ttf",
                         "Alkalami.ttf", "Gulzar.ttf", "NotoNastaliqUrdu.ttf",
                         "Caveat.ttf", "PatrickHand.ttf", "DancingScript.ttf", "IndieFlower.ttf", "ShadowsIntoLight.ttf",
                         "Kalam.ttf", "ArchitectsDaughter.ttf", "GloriaHallelujah.ttf", "NothingYouCouldDo.ttf",
                         "CoveredByYourGrace.ttf", "GochiHand.ttf", "Neucha.ttf", "Pangolin.ttf", "ShortStack.ttf",
                         "SueEllenFrancisco.ttf", "NanumPenScript.ttf", "Schoolbell.ttf", "ComingSoon.ttf", "RockSalt.ttf",
                         "HomemadeApple.ttf", "ReenieBeanie.ttf", "Handlee.ttf"],
    "display": ["Lalezar.ttf", "Katibeh.ttf", "Lemonada.ttf", "Rakkas.ttf", "Jomhuria.ttf", "Blaka.ttf", "ReemKufiFun.ttf",
                "BadeenDisplay.ttf", "Oswald.ttf", "PlayfairDisplay.ttf"],
    "mono": ["CourierPrime.ttf", "IBMPlexMono.ttf", "RobotoMono.ttf", "SourceCodePro.ttf", "Cousine.ttf", "JetBrainsMono.ttf", "SpaceMono.ttf"],
    "heldout_forms": ["Fustat.ttf", "Alkalami.ttf", "PublicSans.ttf", "SpaceMono.ttf", "ReenieBeanie.ttf", "Handlee.ttf"],
}
LATIN_NEEDED = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789.,:;-()/@+%")
ARABIC_NEEDED = ([chr(c) for c in range(0x0621, 0x063B)] + [chr(c) for c in range(0x0641, 0x0653)]
                 + [chr(c) for c in range(0x0660, 0x066A)] + list("،؛؟") + LATIN_NEEDED)
FONT_DIR = os.path.join(DATA_DIR, "fonts")
v5_fonts = os.path.join(V5_DATA_DIR, "fonts")
if os.path.isdir(v5_fonts):                     # v5 fonts are already checked: copy instead of downloading
    for name in os.listdir(v5_fonts):
        if name.endswith(".ttf") and not os.path.exists(os.path.join(FONT_DIR, name)):
            shutil.copy(os.path.join(v5_fonts, name), FONT_DIR)


def fetch_font(name, path, needed):
    target = os.path.join(FONT_DIR, name)
    if not os.path.exists(target):
        try:
            urllib.request.urlretrieve(GOOGLE_FONTS + path, target)
        except Exception as e:
            print(f"   skipped {name}: could not download ({type(e).__name__})")
            return False
    try:
        cmap = TTFont(target).getBestCmap()
    except Exception as e:
        os.remove(target)
        print(f"   skipped {name}: unreadable ({type(e).__name__})")
        return False
    missing = [c for c in needed if ord(c) not in cmap]
    if missing:
        os.remove(target)
        print(f"   skipped {name}: missing {''.join(missing[:10])}")
        return False
    return True


start = step("1. fonts")
fonts = {"arabic": [n for n, p in ARABIC_FONT_FILES.items() if fetch_font(n, p, ARABIC_NEEDED)],
         "english": [n for n, p in ENGLISH_FONT_FILES.items() if fetch_font(n, p, LATIN_NEEDED)]}
fonts.update({k: [n for n in v if n in fonts["arabic"] + fonts["english"]] for k, v in FONT_STYLE.items()})
json.dump(fonts, open(os.path.join(FONT_DIR, "fonts.json"), "w"), indent=1)
note("fonts", arabic=len(fonts["arabic"]), english=len(fonts["english"]), heldout_forms=fonts["heldout_forms"],
     licence="SIL Open Font License; Apache 2.0 for Schoolbell, Coming Soon, Rock Salt, Homemade Apple")
print(f"fonts: {len(fonts['arabic'])} Arabic, {len(fonts['english'])} English; held out for forms: {', '.join(fonts['heldout_forms'])} "
      f"({time.time() - start:.0f} s)")

## 2. Text: sentences to draw, and the vocabulary of the form generator
The Wikipedia and diacritised sentences come from the v5 folder (they were already filtered against KITAB-Bench);
without it they are downloaded as in v5. New: `vocab.json` holds what the form generator v2 and the template filler
draw values from: 100 Arabic male and 94 female first names and 100 family names, each with a Latin spelling, 104
English first names and 109 surnames, places, occupations, and the 40 field types with their Arabic and English
labels. Names are combined at random, so a "person" on a generated form is a random combination of common names, not
anyone in particular.

In [ ]:
def text_file(name):
    return os.path.join(DATA_DIR, "text", name)


def wikipedia_sentences(language, n_articles):
    sentences = []
    for i, row in enumerate(load_dataset("wikimedia/wikipedia", f"20231101.{language}", split="train", streaming=True)):
        if i >= n_articles:
            break
        for s in re.split(r"[.!?؟\n]+", row["text"]):
            s = re.sub(r"\s+", " ", s).strip()
            if 3 <= len(s.split()) <= 60 and not re.search(r"[{}|=<>\[\]]", s):
                sentences.append(s)
    return list(dict.fromkeys(sentences))


start = step("2. text")
for name in ["wikipedia_ar.json", "wikipedia_en.json", "tashkeel_ar.json"]:
    target, source = text_file(name), os.path.join(V5_DATA_DIR, "text", name)
    if os.path.exists(target) and not REDO:
        continue
    if os.path.exists(source):
        shutil.copy(source, target)
        print(f"   {name}: copied from the v5 data folder")
    elif name.startswith("wikipedia"):
        language = name[10:12]
        json.dump(wikipedia_sentences(language, {"ar": 20_000, "en": 6_000}[language] if not QUICK_TEST else 30),
                  open(target, "w", encoding="utf-8"), ensure_ascii=False)
        print(f"   {name}: downloaded (no v5 copy; the training notebook removes sentences that overlap the test suite)")
    else:
        rows = load_dataset("riotu-lab/tashkeel-arabic-sentences", split="train")
        rows = first(rows, 200 if QUICK_TEST else None)
        sentences = [re.sub(r"\s+", " ", s).strip() for s in rows["sentence"]]
        json.dump([s for s in dict.fromkeys(sentences) if 2 <= len(s.split()) <= 40], open(target, "w", encoding="utf-8"), ensure_ascii=False)
        print(f"   {name}: downloaded")
text_counts = {name: len(json.load(open(text_file(name), encoding="utf-8"))) for name in ["wikipedia_ar.json", "wikipedia_en.json", "tashkeel_ar.json"]}

ARABIC_MALE = [("محمد", "Mohammed"), ("أحمد", "Ahmed"), ("خالد", "Khalid"), ("سعيد", "Saeed"), ("عبدالله", "Abdulla"), ("راشد", "Rashid"),
               ("حمدان", "Hamdan"), ("منصور", "Mansoor"), ("سلطان", "Sultan"), ("علي", "Ali"), ("حمد", "Hamad"), ("يوسف", "Yousef"),
               ("عمر", "Omar"), ("فيصل", "Faisal"), ("إبراهيم", "Ibrahim"), ("إسماعيل", "Ismail"), ("عبدالرحمن", "Abdulrahman"),
               ("عبدالعزيز", "Abdulaziz"), ("عبدالكريم", "Abdulkarim"), ("عبدالرحيم", "Abdulrahim"), ("عبدالمجيد", "Abdulmajeed"),
               ("عبدالحميد", "Abdulhamid"), ("ماجد", "Majid"), ("مروان", "Marwan"), ("ناصر", "Nasser"), ("نبيل", "Nabil"), ("نواف", "Nawaf"),
               ("هاني", "Hani"), ("هشام", "Hisham"), ("وليد", "Waleed"), ("ياسر", "Yasser"), ("زايد", "Zayed"), ("طارق", "Tariq"),
               ("عادل", "Adel"), ("عماد", "Imad"), ("عيسى", "Eisa"), ("موسى", "Musa"), ("مصطفى", "Mustafa"), ("محمود", "Mahmoud"),
               ("مراد", "Murad"), ("بدر", "Badr"), ("بلال", "Bilal"), ("جاسم", "Jassim"), ("جمال", "Jamal"), ("حسن", "Hassan"),
               ("حسين", "Hussain"), ("حمزة", "Hamza"), ("خليفة", "Khalifa"), ("خليل", "Khalil"), ("رامي", "Rami"), ("رياض", "Riyadh"),
               ("زياد", "Ziyad"), ("سالم", "Salem"), ("سامي", "Sami"), ("سعود", "Saud"), ("سيف", "Saif"), ("شادي", "Shadi"),
               ("صالح", "Saleh"), ("ضياء", "Diaa"), ("عامر", "Amer"), ("عثمان", "Othman"), ("عدنان", "Adnan"), ("عصام", "Essam"),
               ("غانم", "Ghanim"), ("فارس", "Fares"), ("فهد", "Fahad"), ("قاسم", "Qasim"), ("كريم", "Karim"), ("مالك", "Malik"),
               ("مبارك", "Mubarak"), ("مجدي", "Magdy"), ("مهند", "Muhannad"), ("نادر", "Nader"), ("نزار", "Nizar"), ("هادي", "Hadi"),
               ("هيثم", "Haitham"), ("وائل", "Wael"), ("يحيى", "Yahya"), ("أنس", "Anas"), ("أسامة", "Osama"), ("أيمن", "Ayman"),
               ("باسل", "Basel"), ("تامر", "Tamer"), ("ثامر", "Thamer"), ("حامد", "Hamid"), ("رائد", "Raed"), ("سلمان", "Salman"),
               ("شريف", "Sharif"), ("صقر", "Saqr"), ("طلال", "Talal"), ("عبيد", "Obaid"), ("غيث", "Ghaith"), ("لؤي", "Louay"),
               ("مازن", "Mazen"), ("معاذ", "Muath"), ("نايف", "Naif"), ("هلال", "Hilal"), ("ريان", "Rayan"), ("آدم", "Adam"), ("إياد", "Iyad")]
ARABIC_FEMALE = [("فاطمة", "Fatima"), ("مريم", "Mariam"), ("عائشة", "Aisha"), ("شيخة", "Shaikha"), ("نورة", "Noura"), ("سارة", "Sara"),
                 ("ليلى", "Laila"), ("موزة", "Moza"), ("حصة", "Hessa"), ("آمنة", "Amna"), ("هند", "Hind"), ("لطيفة", "Latifa"),
                 ("خديجة", "Khadija"), ("زينب", "Zainab"), ("رقية", "Ruqaya"), ("سلمى", "Salma"), ("سمية", "Sumaya"), ("أسماء", "Asma"),
                 ("إيمان", "Iman"), ("أمل", "Amal"), ("بشرى", "Bushra"), ("جميلة", "Jamila"), ("حنان", "Hanan"), ("حليمة", "Halima"),
                 ("خلود", "Kholoud"), ("دانة", "Dana"), ("رنا", "Rana"), ("ريم", "Reem"), ("روان", "Rawan"), ("رحاب", "Rehab"),
                 ("زهرة", "Zahra"), ("سعاد", "Suad"), ("سهى", "Suha"), ("شهد", "Shahad"), ("شيماء", "Shaimaa"), ("صفية", "Safiya"),
                 ("عبير", "Abeer"), ("علياء", "Alia"), ("غادة", "Ghada"), ("فرح", "Farah"), ("لبنى", "Lubna"), ("لينا", "Lina"),
                 ("مها", "Maha"), ("منى", "Mona"), ("ميرة", "Meera"), ("ميساء", "Maysaa"), ("نادية", "Nadia"), ("نجلاء", "Najla"),
                 ("نوال", "Nawal"), ("هالة", "Hala"), ("هبة", "Heba"), ("هدى", "Huda"), ("وفاء", "Wafaa"), ("ياسمين", "Yasmin"),
                 ("أروى", "Arwa"), ("بسمة", "Basma"), ("تسنيم", "Tasneem"), ("جنى", "Jana"), ("ديما", "Dima"), ("رغد", "Raghad"),
                 ("سدرة", "Sidra"), ("شروق", "Shurooq"), ("عفراء", "Afraa"), ("غدير", "Ghadeer"), ("لمى", "Lama"), ("ملك", "Malak"),
                 ("نوف", "Nouf"), ("وعد", "Waad"), ("يارا", "Yara"), ("أميرة", "Amira"), ("إسراء", "Israa"), ("آلاء", "Alaa"),
                 ("بتول", "Batool"), ("ثريا", "Thuraya"), ("جود", "Joud"), ("حياة", "Hayat"), ("دعاء", "Doaa"), ("رشا", "Rasha"),
                 ("سناء", "Sanaa"), ("شذى", "Shatha"), ("عزة", "Azza"), ("فدوى", "Fadwa"), ("ليان", "Layan"), ("منار", "Manar"),
                 ("نسرين", "Nesreen"), ("هيفاء", "Haifa"), ("وجدان", "Wijdan"), ("سلوى", "Salwa"), ("ميار", "Mayar"), ("رزان", "Razan"),
                 ("لجين", "Lujain"), ("ريما", "Rima"), ("دلال", "Dalal"), ("عهود", "Ohoud")]
ARABIC_FAMILY = [("الكعبي", "Al Kaabi"), ("المنصوري", "Al Mansoori"), ("الظاهري", "Al Dhaheri"), ("النعيمي", "Al Nuaimi"), ("الشامسي", "Al Shamsi"),
                 ("المزروعي", "Al Mazrouei"), ("الكتبي", "Al Ketbi"), ("السويدي", "Al Suwaidi"), ("المهيري", "Al Muhairi"), ("الهاشمي", "Al Hashemi"),
                 ("البلوشي", "Al Balushi"), ("العامري", "Al Ameri"), ("الحمادي", "Al Hammadi"), ("الزعابي", "Al Zaabi"), ("الشحي", "Al Shehhi"),
                 ("الطنيجي", "Al Tunaiji"), ("العليلي", "Al Alili"), ("الفلاسي", "Al Falasi"), ("القاسمي", "Al Qasimi"), ("المري", "Al Marri"),
                 ("المطروشي", "Al Matrooshi"), ("النقبي", "Al Naqbi"), ("الهاملي", "Al Hamli"), ("الكندي", "Al Kindi"), ("الرميثي", "Al Romaithi"),
                 ("البادي", "Al Badi"), ("الدرعي", "Al Darei"), ("السعدي", "Al Saadi"), ("الشكيلي", "Al Shukaili"), ("الحوسني", "Al Hosani"),
                 ("العبدولي", "Al Abdouli"), ("الجنيبي", "Al Junaibi"), ("البستكي", "Al Bastaki"), ("الخوري", "Al Khoori"), ("الزرعوني", "Al Zarooni"),
                 ("الغفلي", "Al Ghafli"), ("القبيسي", "Al Qubaisi"), ("الكثيري", "Al Kathiri"), ("المرزوقي", "Al Marzooqi"), ("المعمري", "Al Maamari"),
                 ("النيادي", "Al Neyadi"), ("الأحبابي", "Al Ahbabi"), ("الحسيني", "Al Husseini"), ("العلي", "Al Ali"), ("الأحمد", "Al Ahmad"),
                 ("الخطيب", "Al Khatib"), ("الحداد", "Al Haddad"), ("النجار", "Al Najjar"), ("الشريف", "Al Sharif"), ("العمري", "Al Omari"),
                 ("الزهراني", "Al Zahrani"), ("الغامدي", "Al Ghamdi"), ("القحطاني", "Al Qahtani"), ("العتيبي", "Al Otaibi"), ("الشمري", "Al Shammari"),
                 ("الدوسري", "Al Dosari"), ("المالكي", "Al Maliki"), ("الحربي", "Al Harbi"), ("السبيعي", "Al Subaie"), ("العنزي", "Al Anazi"),
                 ("الرشيدي", "Al Rashidi"), ("الخالدي", "Al Khalidi"), ("التميمي", "Al Tamimi"), ("البكري", "Al Bakri"), ("المصري", "Al Masri"),
                 ("الحلبي", "Al Halabi"), ("الشامي", "Al Shami"), ("البغدادي", "Al Baghdadi"), ("المغربي", "Al Maghribi"), ("التونسي", "Al Tunisi"),
                 ("الجزائري", "Al Jazairi"), ("اليمني", "Al Yamani"), ("السوداني", "Al Sudani"), ("عبدالله", "Abdullah"), ("حسن", "Hassan"),
                 ("إبراهيم", "Ibrahim"), ("يوسف", "Yousef"), ("منصور", "Mansour"), ("سليمان", "Suleiman"), ("عثمان", "Othman"), ("خليل", "Khalil"),
                 ("صالح", "Saleh"), ("حمدان", "Hamdan"), ("عيسى", "Eisa"), ("داود", "Dawood"), ("ناصر", "Nasser"), ("عمران", "Imran"),
                 ("بكر", "Bakr"), ("جابر", "Jaber"), ("حجازي", "Hegazy"), ("شاهين", "Shaheen"), ("زيدان", "Zeidan"), ("سلامة", "Salama"),
                 ("عطية", "Attia"), ("نصر", "Nasr"), ("رضوان", "Radwan"), ("بركات", "Barakat"), ("درويش", "Darwish"), ("قاسم", "Qasim"),
                 ("حماد", "Hammad")]
ENGLISH_FIRST = ("James John Robert Michael William David Richard Joseph Thomas Charles Christopher Daniel Matthew Anthony Mark Donald "
                 "Steven Paul Andrew Joshua Kevin Brian George Timothy Ronald Edward Jason Jeffrey Ryan Jacob Gary Nicholas Eric Jonathan "
                 "Stephen Larry Justin Scott Brandon Benjamin Samuel Gregory Alexander Patrick Frank Raymond Jack Dennis Tyler Mary "
                 "Patricia Jennifer Linda Elizabeth Barbara Susan Jessica Sarah Karen Lisa Nancy Sandra Margaret Ashley Kimberly Emily "
                 "Donna Michelle Carol Amanda Melissa Deborah Stephanie Rebecca Sharon Laura Cynthia Amy Kathleen Angela Emma Brenda "
                 "Pamela Nicole Anna Samantha Katherine Christine Rachel Janet Maria Olivia Heather Helen Priya Arjun Rahul Anita Sanjay "
                 "Deepa Ravi Sunita Carlos Sofia").split()
ENGLISH_LAST = ("Smith Johnson Williams Brown Jones Garcia Miller Davis Rodriguez Martinez Hernandez Lopez Gonzalez Wilson Anderson "
                "Thomas Taylor Moore Jackson Martin Lee Perez Thompson White Harris Sanchez Clark Ramirez Lewis Robinson Walker Young "
                "Allen King Wright Scott Torres Nguyen Hill Flores Green Adams Nelson Baker Hall Rivera Campbell Mitchell Carter Roberts "
                "Gomez Phillips Evans Turner Diaz Parker Cruz Edwards Collins Reyes Stewart Morris Morales Murphy Cook Rogers Gutierrez "
                "Ortiz Morgan Cooper Peterson Bailey Reed Kelly Howard Ramos Kim Cox Ward Richardson Watson Brooks Chavez Wood James "
                "Bennett Gray Mendoza Ruiz Hughes Price Alvarez Castillo Sanders Patel Myers Long Ross Foster Jimenez Sharma Singh "
                "Kumar Chen Wang Tanaka Fernandes Menon Pillai").split()
VOCAB = {
    "arabic_male": ARABIC_MALE, "arabic_female": ARABIC_FEMALE, "arabic_family": ARABIC_FAMILY,
    "english_first": ENGLISH_FIRST, "english_last": ENGLISH_LAST,
    "emirates": [("أبوظبي", "Abu Dhabi"), ("دبي", "Dubai"), ("الشارقة", "Sharjah"), ("عجمان", "Ajman"), ("أم القيوين", "Umm Al Quwain"),
                 ("رأس الخيمة", "Ras Al Khaimah"), ("الفجيرة", "Fujairah")],
    "cities": [("العين", "Al Ain"), ("خورفكان", "Khor Fakkan"), ("كلباء", "Kalba"), ("دبا", "Dibba"), ("الذيد", "Al Dhaid"),
               ("مدينة زايد", "Madinat Zayed"), ("الرويس", "Ruwais"), ("الرياض", "Riyadh"), ("جدة", "Jeddah"), ("الدمام", "Dammam"),
               ("مسقط", "Muscat"), ("صلالة", "Salalah"), ("الدوحة", "Doha"), ("المنامة", "Manama"), ("الكويت", "Kuwait City"),
               ("عمّان", "Amman"), ("القاهرة", "Cairo"), ("الإسكندرية", "Alexandria"), ("بيروت", "Beirut"), ("دمشق", "Damascus"),
               ("بغداد", "Baghdad"), ("الدار البيضاء", "Casablanca"), ("تونس", "Tunis"), ("الخرطوم", "Khartoum")],
    "areas": [("البطين", "Al Bateen"), ("الخالدية", "Al Khalidiya"), ("المشرف", "Al Mushrif"), ("الكرامة", "Al Karama"), ("البرشاء", "Al Barsha"),
              ("ديرة", "Deira"), ("الجميرا", "Jumeirah"), ("القصيص", "Al Qusais"), ("المجاز", "Al Majaz"), ("النهدة", "Al Nahda"),
              ("الراشدية", "Al Rashidiya"), ("المويهات", "Al Mowaihat"), ("الحمرية", "Al Hamriyah"), ("الرملة", "Al Ramlah"),
              ("القرم", "Al Qurm"), ("الصفوح", "Al Sufouh")],
    "streets": [("شارع الكورنيش", "Corniche Street"), ("شارع الشيخ زايد", "Sheikh Zayed Road"), ("شارع المطار", "Airport Road"),
                ("شارع خليفة", "Khalifa Street"), ("شارع حمدان", "Hamdan Street"), ("شارع الوحدة", "Al Wahda Street"),
                ("شارع النصر", "Al Nasr Street"), ("شارع السلام", "Al Salam Street"), ("شارع الاتحاد", "Al Ittihad Road"),
                ("شارع الملك فيصل", "King Faisal Street"), ("شارع الجامعة", "University Street"), ("شارع الميناء", "Al Mina Street")],
    "nationalities": [("الإمارات", "United Arab Emirates"), ("السعودية", "Saudi Arabia"), ("مصر", "Egypt"), ("الأردن", "Jordan"), ("الهند", "India"),
                      ("باكستان", "Pakistan"), ("الفلبين", "Philippines"), ("سوريا", "Syria"), ("لبنان", "Lebanon"), ("المغرب", "Morocco"),
                      ("بنغلاديش", "Bangladesh"), ("السودان", "Sudan"), ("عُمان", "Oman"), ("البحرين", "Bahrain"), ("الكويت", "Kuwait"),
                      ("قطر", "Qatar"), ("العراق", "Iraq"), ("اليمن", "Yemen"), ("تونس", "Tunisia"), ("الجزائر", "Algeria"),
                      ("فلسطين", "Palestine"), ("نيبال", "Nepal"), ("سريلانكا", "Sri Lanka"), ("المملكة المتحدة", "United Kingdom"),
                      ("الولايات المتحدة", "United States"), ("كندا", "Canada"), ("أستراليا", "Australia"), ("فرنسا", "France"), ("ألمانيا", "Germany")],
    "occupations": [("مهندس", "Engineer"), ("محاسب", "Accountant"), ("معلم", "Teacher"), ("طبيب", "Doctor"), ("ممرض", "Nurse"),
                    ("محامي", "Lawyer"), ("موظف", "Employee"), ("مدير", "Manager"), ("مدير مشاريع", "Project Manager"), ("فني", "Technician"),
                    ("سائق", "Driver"), ("طالب", "Student"), ("مبرمج", "Software Developer"), ("مصمم", "Designer"), ("مستشار", "Consultant"),
                    ("باحث", "Researcher"), ("صيدلي", "Pharmacist"), ("مندوب مبيعات", "Sales Representative"), ("أمين صندوق", "Cashier"),
                    ("محلل بيانات", "Data Analyst"), ("كهربائي", "Electrician"), ("ميكانيكي", "Mechanic"), ("محاضر", "Lecturer"),
                    ("مترجم", "Translator"), ("رجل أعمال", "Businessman"), ("متقاعد", "Retired")],
    "marital": [("أعزب", "Single"), ("متزوج", "Married"), ("مطلق", "Divorced"), ("أرمل", "Widowed")],
    "gender": [("ذكر", "Male"), ("أنثى", "Female")],
    "relationships": [("أب", "Father"), ("أم", "Mother"), ("أخ", "Brother"), ("أخت", "Sister"), ("عم", "Uncle"), ("جد", "Grandfather"),
                      ("جدة", "Grandmother"), ("زوج", "Husband"), ("زوجة", "Wife"), ("ابن", "Son"), ("ابنة", "Daughter")],
    "school_words": [("النور", "Al Noor"), ("الأمل", "Al Amal"), ("المستقبل", "Al Mustaqbal"), ("الفجر", "Al Fajr"), ("الرواد", "Al Rowad"),
                     ("الإبداع", "Al Ibdaa"), ("الياسمين", "Al Yasmin"), ("الخليج", "Al Khaleej"), ("الريادة", "Al Riyada"), ("القمة", "Al Qimma")],
    "grades": [(f"الصف {w}", f"Grade {i}") for i, w in enumerate(["الأول", "الثاني", "الثالث", "الرابع", "الخامس", "السادس", "السابع",
                                                                    "الثامن", "التاسع", "العاشر", "الحادي عشر", "الثاني عشر"], 1)],
    "company_words": [("النور", "Al Noor"), ("الخليج", "Al Khaleej"), ("الإمارات", "Emirates"), ("المستقبل", "Al Mustaqbal"), ("البركة", "Al Baraka"),
                      ("الصقر", "Al Saqr"), ("النخيل", "Al Nakheel"), ("الأفق", "Al Ufuq"), ("السلام", "Al Salam"), ("الواحة", "Al Waha"),
                      ("الريادة", "Al Riyada"), ("الفجر", "Al Fajr"), ("الأمل", "Al Amal"), ("الرمال", "Al Rimal"), ("البحر", "Al Bahr"),
                      ("الشروق", "Al Shurooq"), ("الجزيرة", "Al Jazira"), ("الهلال", "Al Hilal"), ("الزمرد", "Al Zumurrud"), ("اللؤلؤة", "Al Lulua")],
    "company_kinds": [("للتجارة", "Trading LLC"), ("للمقاولات", "Contracting LLC"), ("للخدمات", "Services LLC"), ("للتقنية", "Technologies FZE"),
                      ("للاستشارات", "Consulting LLC"), ("للنقل", "Transport LLC"), ("للعقارات", "Real Estate LLC"), ("للسياحة", "Tourism LLC")],
    "choices": [[("جديد", "New"), ("تجديد", "Renewal"), ("إلغاء", "Cancellation"), ("تعديل", "Amendment"), ("بدل فاقد", "Replacement")],
                [("فرد", "Individual"), ("شركة", "Company"), ("حكومة", "Government")],
                [("نقدي", "Cash"), ("بطاقة", "Card"), ("تحويل بنكي", "Bank Transfer"), ("شيك", "Cheque")]],
    "titles": [("استمارة طلب خدمة", "Service Request Form"), ("فاتورة", "Invoice"), ("طلب تسجيل", "Registration Application"),
               ("طلب تجديد رخصة", "Licence Renewal Request"), ("إيصال دفع", "Payment Receipt"), ("نموذج بيانات الموظف", "Employee Data Form"),
               ("طلب فتح حساب", "Account Opening Form"), ("نموذج تسجيل طالب", "Student Registration Form"), ("طلب إجازة", "Leave Request Form"),
               ("طلب اشتراك", "Subscription Form"), ("نموذج بيانات المستأجر", "Tenant Information Form"), ("نموذج شكوى", "Complaint Form"),
               ("طلب صيانة", "Maintenance Request"), ("نموذج تحديث بيانات", "Data Update Form"), ("طلب عضوية", "Membership Application"),
               ("إشعار استلام", "Delivery Note")],
    "sections": [("بيانات مقدم الطلب", "Applicant Details"), ("بيانات الاتصال", "Contact Details"), ("بيانات العمل", "Employment Details"),
                 ("بيانات الدفع", "Payment Details"), ("معلومات إضافية", "Additional Information"), ("للاستخدام الرسمي", "For Official Use"),
                 ("بيانات الطالب", "Student Details"), ("بيانات ولي الأمر", "Guardian Details")],
    "months_ar": ["يناير", "فبراير", "مارس", "أبريل", "مايو", "يونيو", "يوليو", "أغسطس", "سبتمبر", "أكتوبر", "نوفمبر", "ديسمبر"],
    "months_en": ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"],
    "email_domains": ["example.com", "example.ae", "example.org", "example.net", "mail.example", "company.example"],
}
FIELD_TYPES = {   # JSON key: (value kind, Arabic labels, English labels). 40 field types; v5 had 12.
    "full_name": ("name_ar", ["الاسم الكامل", "الاسم", "اسم مقدم الطلب", "الاسم الثلاثي"], ["Full Name", "Name", "Applicant Name"]),
    "full_name_en": ("name_en", ["الاسم بالإنجليزية", "الاسم (بالإنجليزية)"], ["Name in English", "Full Name (English)"]),
    "date": ("date", ["التاريخ", "تاريخ الطلب"], ["Date", "Application Date"]),
    "date_of_birth": ("dob", ["تاريخ الميلاد"], ["Date of Birth", "DOB"]),
    "place_of_birth": ("city", ["مكان الميلاد"], ["Place of Birth"]),
    "issue_date": ("past_date", ["تاريخ الإصدار"], ["Issue Date", "Date of Issue"]),
    "expiry_date": ("future_date", ["تاريخ الانتهاء", "تاريخ انتهاء الصلاحية"], ["Expiry Date", "Valid Until"]),
    "phone": ("phone", ["رقم الهاتف", "الهاتف"], ["Phone", "Telephone"]),
    "mobile": ("mobile", ["رقم الجوال", "الهاتف المتحرك"], ["Mobile", "Mobile No."]),
    "fax": ("phone", ["الفاكس"], ["Fax"]),
    "email": ("email", ["البريد الإلكتروني"], ["Email", "E-mail Address"]),
    "po_box": ("po_box", ["صندوق البريد", "ص.ب"], ["P.O. Box"]),
    "emirate": ("emirate", ["الإمارة"], ["Emirate"]),
    "city": ("city", ["المدينة"], ["City"]),
    "area": ("area", ["المنطقة", "الحي"], ["Area", "District"]),
    "address": ("address", ["العنوان", "عنوان السكن"], ["Address", "Home Address"]),
    "nationality": ("nationality", ["الجنسية"], ["Nationality"]),
    "gender": ("gender", ["الجنس", "النوع"], ["Gender", "Sex"]),
    "marital_status": ("marital", ["الحالة الاجتماعية"], ["Marital Status"]),
    "occupation": ("occupation", ["المهنة", "الوظيفة"], ["Occupation", "Job Title"]),
    "employer": ("company", ["جهة العمل", "اسم جهة العمل"], ["Employer"]),
    "company": ("company", ["اسم الشركة", "اسم المنشأة"], ["Company", "Company Name"]),
    "trade_licence": ("licence", ["رقم الرخصة التجارية", "رقم الرخصة"], ["Trade Licence No.", "Licence No."]),
    "reference_number": ("reference", ["رقم الطلب", "الرقم المرجعي", "رقم المرجع"], ["Reference No.", "Ref. No."]),
    "invoice_number": ("invoice", ["رقم الفاتورة"], ["Invoice No."]),
    "emirates_id": ("emirates_id", ["رقم الهوية", "رقم الهوية الإماراتية"], ["Emirates ID", "ID Number"]),
    "passport_number": ("passport", ["رقم الجواز", "رقم جواز السفر"], ["Passport No."]),
    "amount_aed": ("amount", ["المبلغ (درهم)", "المبلغ", "قيمة المبلغ"], ["Amount (AED)", "Amount"]),
    "vat_amount": ("amount_small", ["ضريبة القيمة المضافة", "قيمة الضريبة"], ["VAT", "VAT Amount"]),
    "total_amount": ("amount", ["المجموع", "الإجمالي", "المبلغ الإجمالي"], ["Total", "Total Amount"]),
    "quantity": ("quantity", ["الكمية", "العدد"], ["Quantity", "Qty"]),
    "student_name": ("name_ar", ["اسم الطالب"], ["Student Name"]),
    "school": ("school", ["اسم المدرسة", "المدرسة"], ["School"]),
    "grade": ("grade", ["الصف", "المرحلة الدراسية"], ["Grade", "Class"]),
    "guardian_name": ("name_ar", ["اسم ولي الأمر"], ["Guardian Name"]),
    "relationship": ("relationship", ["صلة القرابة"], ["Relationship"]),
    "vehicle_plate": ("plate", ["رقم اللوحة", "رقم لوحة المركبة"], ["Plate No.", "Vehicle Plate"]),
    "service_type": ("choice", ["نوع الخدمة", "نوع الطلب"], ["Service Type", "Request Type"]),
    "notes": ("free_text", ["ملاحظات", "ملاحظات إضافية"], ["Notes", "Remarks", "Comments"]),
    "reason": ("free_text", ["سبب الطلب"], ["Reason for Request"]),
}
VOCAB["field_types"] = {k: {"kind": kind, "ar": ar, "en": en} for k, (kind, ar, en) in FIELD_TYPES.items()}
json.dump(VOCAB, open(text_file("vocab.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=0)
note("text", **text_counts, vocab_field_types=len(FIELD_TYPES),
     names=len(ARABIC_MALE) + len(ARABIC_FEMALE) + len(ARABIC_FAMILY) + len(ENGLISH_FIRST) + len(ENGLISH_LAST),
     licences={"wikipedia": "CC-BY-SA", "tashkeel": "CC (riotu-lab, from Alukah)", "vocab": "written for this project"})
print(f"text: {text_counts}; vocab.json with {len(FIELD_TYPES)} field types ({time.time() - start:.0f} s)")

## 3. Download every source
Each source is saved as it comes, in one of two places: **test splits** go to `datasets/` (they are the test sets and
are never changed), **training splits** go to `raw/` (section 5 filters them against the test suite before anything
trains on them). Every line dataset has the same columns: `image`, `text`, `source`, `writer` (a writer or form ID
where the source has one), `group` (a page, book or paragraph, used to keep related lines on one side of a split) and
`note`.

| Source | Train | Test | Terms | Notes |
| --- | --- | --- | --- | --- |
| KITAB-Bench, 9 line subsets | – | yes | per subset | copied from v5 |
| Teklia/KHATT_v1.0 | train + validation | test | research | copied from v5; no writer IDs |
| official KHATT (optional) | train | validation, test, seen-writer test | research, on request | writer = form ID |
| aamijar/muharaf-public | train | validation, test | CC BY-NC-SA 4.0 | copied from v5 |
| Nakba memoir lines (gated) | train | test (blind test kept for self-labelling) | CC BY 4.0 | one collection |
| Archatext/AraMS-28k-HTR | 9 books | 2 books validation, 3 books test | CC BY-NC-SA 4.0 | split by book |
| Teklia/IAM-line | train | validation, test | non-commercial research | |
| GNHK (Hugging Face mirror) | train | test | CC BY 4.0 | lines built from word boxes |
| olmOCR-mix-1025 | optional, 1 archive | evaluation pages | ODC-BY | labels written by GPT-4.1 |
| FUNSD | – | 50 forms | research use; check the FUNSD page | real scanned forms |
| SynthDoG-en (section 5) | 20,000 pages | – | not stated on the card | rendered English pages, checked against the suite as they download |

In [ ]:
start = step("3a. sources already in the v5 folder (copied), or downloaded")
KITAB_SUBSETS = {"PATS": "ahmedheakl/arocrbench_patsocr", "SythenAR": "ahmedheakl/arocrbench_synthesizear",
                 "ISI-PPT": "ahmedheakl/arocrbench_isippt", "KHATT": "ahmedheakl/arocrbench_khatt",
                 "OnlineKhatt": "ahmedheakl/arocrbench_onlinekhatt", "EvArEST": "ahmedheakl/arocrbench_evarest",
                 "HistoryAr": "ahmedheakl/arocrbench_historyar", "ADAB": "ahmedheakl/arocrbench_adab",
                 "Muharaf": "ahmedheakl/arocrbench_muharaf"}
for name, repo in KITAB_SUBSETS.items():
    if copy_v5(f"kitab_{name}") is None:
        try:
            ds = load_dataset(repo, split="train")
            if text_column(ds) != "text":
                ds = ds.rename_column(text_column(ds), "text")
            save(first(ds, LIMIT), f"kitab_{name}")
            print(f"   kitab_{name}: downloaded")
        except Exception as e:
            print(f"   KITAB {name}: skipped ({type(e).__name__}: {str(e)[:80]})")


def load_iam(split):
    """IAM lines (train 6,482 / validation 976 / test 2,915). Newer `datasets` versions refuse a single-split
    `data_files` load (ExpectedMoreSplitsError), so the repository is loaded whole; the parquet file is the fallback."""
    try:
        return load_dataset("Teklia/IAM-line", split=split)
    except Exception as e:
        print(f"   IAM: plain load failed ({type(e).__name__}); loading the parquet file directly")
        return load_dataset("Teklia/IAM-line", data_files={split: f"data/{split}.parquet"}, split=split, verification_mode="no_checks")


# (v5 name, v6 name, folder, how to download it if there is no v5 copy)
V5_SOURCES = [
    ("khatt_train", "khatt_teklia_train", "raw",
     lambda: concatenate_datasets([first(load_dataset("Teklia/KHATT_v1.0", split=s), LIMIT) for s in ["train", "validation"]])),
    ("khatt_test", "khatt_test", "datasets", lambda: first(load_dataset("Teklia/KHATT_v1.0", split="test"), LIMIT)),
    ("muharaf_train", "muharaf_train", "raw", lambda: first(load_dataset("aamijar/muharaf-public", split="train"), LIMIT)),
    ("muharaf_validation", "muharaf_validation", "datasets", lambda: first(load_dataset("aamijar/muharaf-public", split="validation"), LIMIT)),
    ("muharaf_test", "muharaf_test", "datasets", lambda: first(load_dataset("aamijar/muharaf-public", split="test"), LIMIT)),
    ("iam_train", "iam_train", "raw", lambda: first(load_iam("train"), LIMIT)),
    ("iam_test", "iam_test", "datasets", lambda: first(load_iam("test"), LIMIT)),
    (None, "iam_validation", "datasets", lambda: first(load_iam("validation"), LIMIT)),     # new in v6: for progress checks
]
for v5_name, name, folder, download in V5_SOURCES:
    if folder == "raw":                          # training sources: read from the v5 folder in place when it has them
        ds = raw_dataset(name)
        if ds is not None:
            print(f"   {name}: {'already saved' if saved_dataset(name, 'raw') is not None else 'read in place from the v5 folder'} ({len(ds):,} lines)")
            continue
    else:                                        # test sets: copied, so the suite lives in this folder
        ds = copy_v5(v5_name, folder, new_name=name) if v5_name else saved_dataset(name, folder)
    if ds is None:
        try:
            ds = download()
            if text_column(ds) != "text":
                ds = ds.rename_column(text_column(ds), "text")
            ds = ds.filter(lambda t: bool((t or "").strip()), input_columns=["text"])
            ds = save(ds, name, folder)
            print(f"   {name}: downloaded ({len(ds):,} lines)")
        except Exception as e:
            print(f"   {name}: skipped ({type(e).__name__}: {str(e)[:100]})")
note("v5_sources", reused_from_v5=os.path.isdir(V5_DATA_DIR),
     licences={"KITAB-Bench": "see each subset", "KHATT": "research use (KHATT terms)", "Muharaf": "CC BY-NC-SA 4.0",
               "IAM": "non-commercial research"})
print(f"   done ({(time.time() - start) / 60:.1f} min)")

### 3b. The official KHATT release (optional)
Our Teklia copy of KHATT has no writer IDs, so it cannot say whether a test writer also wrote training lines, nor
which lines are the fixed passages every writer copied. The official release can. **To use it:** request it from
KFUPM at [khatt.ideas2serve.net](http://khatt.ideas2serve.net/), then put what you receive (zip or rar archives, or the
unpacked folders) in `MyDrive/arabic_vlm/khatt_official/` and run this cell again.

The reader below walks every folder, takes **line** images that have a ground-truth `.txt` file with the same name,
and reads from the folder names whether a line is fixed or unique text (`Fixed...`/`Unique...`) and which split it
belongs to (`Train`/`Validat...`/`Test`). The writer is the form ID at the start of the file name (KHATT has one
form per writer). It prints the first file names and what it read from them, so you can check the layout matched.

It also builds the **seen-writer test**: one unique-text paragraph from each of 100 training writers is held out
(removed from training), so seen-writer CER can be compared with unseen-writer CER on the official test split.

In [ ]:
IMAGE_EXTS = (".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp")


def unpack_archives(folder, label):
    """Unpack every zip or rar file under `folder` once, into the local work folder. Returns the folders to read."""
    out = os.path.join(WORK_DIR, label)
    os.makedirs(out, exist_ok=True)
    for path in sorted(glob.glob(os.path.join(folder, "**", "*"), recursive=True)):
        low = path.lower()
        if not low.endswith((".zip", ".rar")):
            continue
        target = os.path.join(out, re.sub(r"\W+", "_", os.path.relpath(path, folder)))
        if os.path.exists(target + ".done"):
            continue
        print(f"   unpacking {os.path.basename(path)}")
        if low.endswith(".zip"):
            zipfile.ZipFile(path).extractall(target)
        else:
            if shutil.which("unrar") is None:
                subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True, check=False)
            os.makedirs(target, exist_ok=True)
            subprocess.run(["unrar", "x", "-o+", "-inul", path, target + "/"], check=True)
        open(target + ".done", "w").close()
    return [folder, out]


def read_text_file(path):
    raw = open(path, "rb").read()
    if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
        text = raw.decode("utf-16")
    else:
        try:
            text = raw.decode("utf-8-sig")
        except UnicodeDecodeError:
            text = raw.decode("cp1256", errors="replace")         # KHATT's older files are in the Windows Arabic code page
    return re.sub(r"\s+", " ", text).strip()


def split_of(dirs):
    for d in reversed(dirs):
        if d.startswith("train"):
            return "train"
        if d.startswith(("valid", "val")):
            return "validation"
        if d.startswith("test"):
            return "test"
    return "unknown"


def parse_khatt_official(roots):
    images, texts = [], {}
    for root in roots:
        for path in glob.glob(os.path.join(root, "**", "*"), recursive=True):
            low = path.lower()
            if low.endswith(IMAGE_EXTS):
                images.append(path)
            elif low.endswith(".txt"):
                texts.setdefault(os.path.splitext(os.path.basename(path))[0].lower(), path)
    rows, skipped = [], Counter()
    for path in sorted(set(images)):
        dirs = [d.lower() for d in os.path.normpath(os.path.dirname(path)).split(os.sep)]
        if any("paragraph" in d for d in dirs) and not any("line" in d for d in dirs):
            skipped["paragraph image"] += 1                  # the release also has whole paragraphs; lines are used
            continue
        stem = os.path.splitext(os.path.basename(path))[0]
        txt = texts.get(stem.lower())
        if txt is None:
            skipped["no ground-truth file"] += 1
            continue
        text = read_text_file(txt)
        if not text:
            skipped["empty text"] += 1
            continue
        kind = "fixed" if any("fixed" in d for d in dirs) else "unique" if any("unique" in d for d in dirs) else "unknown"
        para = re.search(r"para(\d+)", stem, re.I)
        form = stem.split("_")[0]
        rows.append({"path": path, "text": text, "split": split_of(dirs), "kind": kind, "writer": form,
                     "group": f"{form}_{'P' + para.group(1) if para else stem}", "file": stem})
    return rows, skipped


def infer_kinds(rows):
    """If the folders do not say fixed or unique: a paragraph is fixed text when most of its 4-grams also occur in the
    paragraphs of at least 5 other writers (every writer copied the fixed passages)."""
    writers_of = defaultdict(set)
    for r in rows:
        for g in ngrams(r["text"]):
            writers_of[g].add(r["writer"])
    by_group = defaultdict(list)
    for r in rows:
        by_group[r["group"]].append(r)
    for group, members in by_group.items():
        grams = set().union(*(ngrams(r["text"]) for r in members))
        shared = sum(len(writers_of[g]) >= 6 for g in grams) / max(1, len(grams))
        for r in members:
            r["kind"] = "fixed" if shared > 0.5 else "unique"


start = step("3b. official KHATT release")
KHATT_OFFICIAL = os.path.isdir(KHATT_OFFICIAL_DIR) and any(os.scandir(KHATT_OFFICIAL_DIR))
if not KHATT_OFFICIAL:
    print(f"   not found at {KHATT_OFFICIAL_DIR} (optional: see the text above). Phase 2 then uses the capped Teklia copy.")
elif saved_dataset("khatt_official_test") is not None and saved_dataset("khatt_official_train", "raw") is not None:
    print("   already prepared")
else:
    rows, skipped = parse_khatt_official(unpack_archives(KHATT_OFFICIAL_DIR, "khatt_official"))
    print(f"   {len(rows):,} line images with text; skipped: {dict(skipped)}")
    for r in rows[:5]:
        print(f"      {r['file']}: split {r['split']}, {r['kind']} text, writer {r['writer']}, paragraph {r['group']}")
    if rows and all(r["kind"] == "unknown" for r in rows):
        infer_kinds(rows)
        print("   the folder names do not say fixed or unique text: inferred from repeated 4-grams across writers")
    if rows and all(r["split"] == "unknown" for r in rows):
        print("   no Train/Validate/Test folders found: splitting by writer 70/15/15 (seeded), as the KHATT paper does by form")
        writers = sorted({r["writer"] for r in rows})
        random.Random(0).shuffle(writers)
        cut1, cut2 = int(0.7 * len(writers)), int(0.85 * len(writers))
        split_of_writer = {w: "train" if i < cut1 else "validation" if i < cut2 else "test" for i, w in enumerate(writers)}
        for r in rows:
            r["split"] = split_of_writer[r["writer"]]
    counts = Counter((r["split"], r["kind"]) for r in rows)
    print("   lines by split and text type:", {f"{s}/{k}": n for (s, k), n in sorted(counts.items())})
    # the seen-writer test: one unique-text paragraph from each of SEEN_WRITERS training writers
    train_unique = defaultdict(lambda: defaultdict(list))
    for i, r in enumerate(rows):
        if r["split"] == "train" and r["kind"] == "unique":
            train_unique[r["writer"]][r["group"]].append(i)
    eligible = sorted(w for w, groups in train_unique.items() if len(groups) >= 2)
    chosen = random.Random(1).sample(eligible, min(SEEN_WRITERS, len(eligible)))
    held = set()
    for w in chosen:
        held.update(train_unique[w][sorted(train_unique[w])[-1]])     # the writer's last unique paragraph
    for i in held:
        rows[i]["split"] = "seen_writer_test"
    print(f"   seen-writer test: {len(held):,} lines from {len(chosen)} training writers (removed from training)")
    for split, name, folder in [("train", "khatt_official_train", "raw"), ("validation", "khatt_official_validation", "datasets"),
                                ("test", "khatt_official_test", "datasets"), ("seen_writer_test", "khatt_seen_writer_test", "datasets")]:
        writer = ShardWriter(name, LINE_FEATURES)
        for r in rows:
            if r["split"] == split:
                with Image.open(r["path"]) as img:
                    img = img.convert("L") if img.mode in ("1", "L", "P", "I;16") else img.convert("RGB")
                    writer.add(line_row(img, r["text"], "khatt_official", r["writer"], r["group"], r["kind"]))
        ds = save(writer.finish(), name, folder)
        print(f"   {name}: {len(ds):,} lines, {len(set(ds['writer']))} writers")
    note("khatt_official", lines=len(rows), by_split_kind={f"{s}/{k}": n for (s, k), n in counts.items()},
         seen_writer_test={"writers": len(chosen), "lines": len(held)}, licence="KHATT research terms (on request from KFUPM)")
print(f"   ({(time.time() - start) / 60:.1f} min)")

### 3c. Nakba memoir lines (gated) and 3d. AraMS-28k
**Memoirs.** 15,969 training and 2,095 test lines from the handwritten memoirs of one author (1951–1965), the data of
the NAKBA NLP 2026 manuscript task. All lines come from one collection and largely one hand, so the test split is
reported as a *seen collection*, never as unseen handwriting. The blind-test lines are kept only as unlabelled lines
for the optional self-labelling step.

**AraMS-28k.** 28,600 lines from 14 manuscript books in Naskh, Ruq'ah and Maghrebi hands. Its fixed split is by book:
9 books train, 2 validation, 3 test, so its test split is the **unseen-book** test the roadmap asks for. Its
transcriptions keep only Arabic letters and spaces (no diacritics, digits or punctuation), so the training notebook
scores it after reducing every model's output the same way, and trains on it with a "letters only" instruction.
Lines are re-saved as JPEG (quality 90) to keep the folder small.

In [ ]:
MEMOIR_REPO = "U4RASD/omar-al-saleh-manuscripts-segments"


def page_key(filename):
    """A page ID from a line image's file name: the name without its trailing line number."""
    stem = os.path.splitext(os.path.basename(filename or ""))[0]
    key = re.sub(r"[_\-\s]*(line|ln|seg|segment|l)?[_\-\s]*\d+$", "", stem, flags=re.I)
    return key or stem


def as_lines(ds, source, group_of=None, writer_of=None, note_of=None):
    """Any image+text dataset in the common line layout, without decoding or re-encoding the images."""
    column = text_column(ds)
    stored = ds.cast_column("image", ImageFeature(decode=False))

    def convert(row):
        return {"text": re.sub(r"\s+", " ", row[column] or "").strip(), "source": source,
                "writer": writer_of(row) if writer_of else "", "group": group_of(row) if group_of else "",
                "note": note_of(row) if note_of else ""}

    out = stored.map(convert, remove_columns=[c for c in stored.column_names if c != "image"], load_from_cache_file=False)
    return out.cast(Features({**LINE_FEATURES, "image": ImageFeature(decode=False)})).cast_column("image", ImageFeature())


start = step("3c. memoir lines")
if not USE_MEMOIRS:
    print("   skipped (USE_MEMOIRS = False)")
for split, name, folder in [("train", "memoir_train", "raw"), ("test", "memoir_test", "datasets"), ("blind_test", "memoir_blind", "datasets")]:
    if not USE_MEMOIRS or saved_dataset(name, folder) is not None:
        continue
    try:
        ds = first(load_dataset(MEMOIR_REPO, split=split, token=HF_TOKEN), LIMIT)
        has_file = "filename" in ds.column_names
        ds = as_lines(ds, "memoir", group_of=(lambda r: page_key(r["filename"])) if has_file else None,
                      note_of=(lambda r: r["filename"] or "") if has_file else None)
        if split != "blind_test":
            ds = ds.filter(lambda t: bool(t), input_columns=["text"])
        ds = save(ds, name, folder)
        print(f"   {name}: {len(ds):,} lines from {len(set(ds['group']))} pages")
    except Exception as e:
        print(f"   {name}: skipped ({type(e).__name__}: {str(e)[:150]}).\n   It is gated: accept its conditions on "
              f"https://huggingface.co/datasets/{MEMOIR_REPO} while logged in, and set the Colab secret HF_TOKEN.")
        break
if USE_MEMOIRS:
    note("memoirs", repo=MEMOIR_REPO, licence="CC BY 4.0 (gated)", seen_collection=True)

start = step("3d. AraMS-28k")
ARAMS_REPO = "Archatext/AraMS-28k-HTR"
if not USE_ARAMS:
    print("   skipped (USE_ARAMS = False)")
elif all(saved_dataset(n, f) is not None for n, f in [("arams_train", "raw"), ("arams_val", "datasets"), ("arams_test", "datasets")]):
    print("   already prepared")
else:
    try:
        archive = hf_file(ARAMS_REPO, "AraMS-28k-HTR.zip")
        folder = os.path.join(WORK_DIR, "arams")
        if not os.path.exists(folder + ".done"):
            zipfile.ZipFile(archive).extractall(folder)
            open(folder + ".done", "w").close()
        metadata = glob.glob(os.path.join(folder, "**", "metadata.csv"), recursive=True)
        assert metadata, "metadata.csv not found in the AraMS-28k-HTR archive"
        files = {os.path.basename(p): p for p in glob.glob(os.path.join(folder, "**", "*.png"), recursive=True)}
        records = list(csv.DictReader(open(metadata[0], encoding="utf-8")))
        for split, name, dest in [("train", "arams_train", "raw"), ("val", "arams_val", "datasets"), ("test", "arams_test", "datasets")]:
            writer, missing = ShardWriter(name, LINE_FEATURES), 0
            for r in [r for r in records if r["split"] == split][:LIMIT]:
                path = files.get(os.path.basename(r["img_path"].replace("\\", "/")))
                if path is None or not r["gt_text"].strip():
                    missing += 1
                    continue
                with Image.open(path) as img:
                    writer.add(line_row(img.convert("L"), r["gt_text"].strip(), "arams", "", r["book_id"],
                                        f"{r['line_type']}|{r['page_id']}", fmt="JPEG"))
            ds = save(writer.finish(), name, dest)
            print(f"   {name}: {len(ds):,} lines from {len(set(ds['group']))} books ({missing} without image or text)")
        note("arams", repo=ARAMS_REPO, licence="CC BY-NC-SA 4.0", split="by book: 9 train / 2 validation / 3 test",
             transcription="Arabic letters and spaces only")
    except Exception as e:
        print(f"   AraMS-28k: skipped ({type(e).__name__}: {str(e)[:150]})")
print(f"   ({(time.time() - start) / 60:.1f} min)")

### 3e. GNHK: English handwriting in phone photos
GNHK (GoodNotes, CC BY 4.0) has 687 phone photos of handwritten notes with a box for every word, the line each word
belongs to (`line_idx`) and whether it is handwritten (`H`) or printed (`P`). Lines are cut out here: the words of a
line are joined left to right, the line's slant is measured from its word boxes and straightened, and lines with
printed words, math, scribbles or unreadable words (`%math%`, `%SC%`, `%NA%`) are skipped. The copy used is a
Hugging Face mirror of the original release (train 515 / test 172 images). 5% of the training photos are kept apart
for progress checks in section 5.

In [ ]:
GNHK_REPO = "aryaman27/goodnotes-handwritten-dataset"


def quad(polygon):
    if isinstance(polygon, dict):
        return [(float(polygon[f"x{i}"]), float(polygon[f"y{i}"])) for i in range(4)]
    flat = np.array(polygon, dtype=float).flatten()
    return list(zip(flat[0::2], flat[1::2]))[:4]


def upright_crop(img, points, pad_ratio=0.2):
    """Crop the region around `points`, rotated so the line they belong to is horizontal."""
    pts = np.array(points, dtype=np.float64)
    x0, y0 = pts.min(0)
    x1, y1 = pts.max(0)
    margin = 0.5 * math.hypot(x1 - x0, y1 - y0) + 10
    bx0, by0 = max(0, int(x0 - margin)), max(0, int(y0 - margin))
    bx1, by1 = min(img.width, int(x1 + margin)), min(img.height, int(y1 + margin))
    patch = img.crop((bx0, by0, bx1, by1))
    pts = pts - [bx0, by0]
    centre = pts.mean(0)
    theta = 0.0
    if len(pts) >= 3:
        values, vectors = np.linalg.eigh(np.cov((pts - centre).T))
        d = vectors[:, np.argmax(values)]
        theta = math.atan2(d[1], d[0])
        theta = theta - math.pi if theta > math.pi / 2 else theta + math.pi if theta < -math.pi / 2 else theta
        if abs(theta) > math.radians(40):                  # a short word's box: no reliable slant
            theta = 0.0
    edge = tuple(int(v) for v in np.median(np.asarray(patch.convert("RGB")).reshape(-1, 3), axis=0))
    rotated = patch.rotate(math.degrees(theta), resample=Image.BICUBIC, center=(float(centre[0]), float(centre[1])), fillcolor=edge)
    turn = np.array([[math.cos(theta), math.sin(theta)], [-math.sin(theta), math.cos(theta)]])
    q = (pts - centre) @ turn.T + centre
    qx0, qy0 = q.min(0)
    qx1, qy1 = q.max(0)
    pad = pad_ratio * max(8.0, qy1 - qy0)
    return rotated.crop((int(max(0, qx0 - pad)), int(max(0, qy0 - pad)), int(min(rotated.width, qx1 + pad)), int(min(rotated.height, qy1 + pad))))


def gnhk_lines(img, words):
    """(crop, text) for every clean handwritten line of one photo."""
    by_line = defaultdict(list)
    for w in words:
        if isinstance(w, dict) and w.get("line_idx") is not None:
            by_line[w["line_idx"]].append(w)
    out = []
    for _, ws in sorted(by_line.items(), key=lambda kv: kv[0]):
        texts = [str(w.get("text", "")).strip() for w in ws]
        if any(str(w.get("type", "H")).upper() != "H" for w in ws) or not all(texts):
            continue
        if any(t.startswith("%") and t.endswith("%") for t in texts):
            continue
        quads = [quad(w["polygon"]) for w in ws]
        order = sorted(range(len(ws)), key=lambda i: min(p[0] for p in quads[i]))
        crop = upright_crop(img, [p for q in quads for p in q])
        if crop.width >= 16 and crop.height >= 8:
            out.append((crop, " ".join(texts[i] for i in order)))
    return out


start = step("3e. GNHK")
if not USE_GNHK:
    print("   skipped (USE_GNHK = False)")
elif saved_dataset("gnhk_train", "raw") is not None and saved_dataset("gnhk_test") is not None:
    print("   already prepared")
else:
    try:
        folder = hf_folder(GNHK_REPO)
        for split, name, dest in [("train", "gnhk_train", "raw"), ("test", "gnhk_test", "datasets")]:
            jsons = sorted(p for p in glob.glob(os.path.join(folder, "**", "*.json"), recursive=True)
                           if f"{split}_data" in p.replace("\\", "/").split("/") or f"/{split}/" in p.replace("\\", "/"))
            writer, photos = ShardWriter(name, LINE_FEATURES), 0
            for jpath in jsons[:LIMIT]:
                stem = os.path.splitext(jpath)[0]
                image = next((stem + e for e in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG") if os.path.exists(stem + e)), None)
                if image is None:
                    continue
                words = json.load(open(jpath, encoding="utf-8"))
                with Image.open(image) as img:
                    img = ImageOps.exif_transpose(img).convert("RGB")
                    for crop, text in gnhk_lines(img, words if isinstance(words, list) else words.get("words", [])):
                        writer.add(line_row(crop, text, "gnhk", "", os.path.basename(stem), fmt="JPEG"))
                photos += 1
            ds = save(writer.finish(), name, dest)
            print(f"   {name}: {len(ds):,} lines from {photos} photos")
            if photos == 0:
                print(f"   no photo+JSON pairs found for {split}; files in the download: {os.listdir(folder)[:10]}")
        note("gnhk", repo=GNHK_REPO, licence="CC BY 4.0 (GNHK, GoodNotes)", lines="built from word boxes and line_idx")
    except Exception as e:
        print(f"   GNHK: skipped ({type(e).__name__}: {str(e)[:150]})")
print(f"   ({(time.time() - start) / 60:.1f} min)")

### 3f. olmOCR-mix pages and 3g. FUNSD forms
**olmOCR-mix-1025** (Ai2, ODC-BY) pairs PDF pages with their text. Its evaluation archive (191 MB) gives the
printed-English *page* test: English pages without tables or diagrams, with at most 400 words, rendered at 1,600 px on
the long side. Its labels were written by GPT-4.1, so spot-check a few before trusting small differences. Markdown
marks (headings, bold, links) are removed from the text for scoring. `USE_OLMOCR_TRAIN = True` also takes one
training archive (about 1 GB, a few thousand pages) as printed-English training pages for phase 4.

**FUNSD** (50 test forms, real scanned English business forms from the 1980s–90s) is the one public set of real
forms with key–value links. Each linked question becomes a JSON key and its answer the value; forms keep at most 12
keys. Its page states research and educational use: check it before any other use.

In [ ]:
OLMOCR_REPO = "allenai/olmOCR-mix-1025"
OLMOCR_SUBSETS = ["00_documents", "01_books", "02_loc_transcripts", "03_national_archives"]
PAGE_FEATURES = Features({"image": ImageFeature(), "text": Value("string"), "source": Value("string"), "writer": Value("string"),
                          "group": Value("string"), "note": Value("string")})


def strip_markdown(text):
    text = re.sub(r"<[^>]+>", " ", text or "")
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)
    text = re.sub(r"\[([^\]]*)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"^\s{0,3}#{1,6}\s*", "", text, flags=re.M)
    text = re.sub(r"(\*\*|__|`)", "", text)
    text = re.sub(r"(?<!\w)[*_](?=\S)|(?<=\S)[*_](?!\w)", "", text)
    text = re.sub(r"^\s*[-*+]\s+", "", text, flags=re.M)
    text = text.replace("$", "")
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)


def render_pdf_page(path, page_number, long_side=1600):
    import pypdfium2 as pdfium
    pdf = pdfium.PdfDocument(path)
    page = pdf[min(max(0, page_number - 1), len(pdf) - 1) if len(pdf) > 1 else 0]
    w, h = page.get_size()
    img = page.render(scale=long_side / max(w, h)).to_pil().convert("RGB")
    pdf.close()
    return img


def olmocr_pages(subset, split, name, folder, max_pages, max_words):
    import pandas as pd
    table = pd.read_parquet(hf_file(OLMOCR_REPO, f"{subset}_{split}.parquet"))
    archive = hf_file(OLMOCR_REPO, f"pdf_tarballs/{subset}_{split}_00000.tar.gz")
    out = os.path.join(WORK_DIR, "olmocr", f"{subset}_{split}")
    if not os.path.exists(out + ".done"):
        with tarfile.open(archive) as tar:
            try:
                tar.extractall(out, filter="data")
            except TypeError:                                  # Python before 3.12
                tar.extractall(out)
        open(out + ".done", "w").close()
    pdfs = {}
    for p in glob.glob(os.path.join(out, "**", "*.pdf"), recursive=True):
        rel = os.path.relpath(p, out).replace("\\", "/")
        pdfs[rel], pdfs[os.path.basename(p)] = p, p
    flag = lambda column, default: table[column].fillna(default).astype(bool)
    keep = table[(table["primary_language"] == "en") & table["natural_text"].notna() & flag("is_rotation_valid", True)
                 & ~flag("is_table", False) & ~flag("is_diagram", False)]
    writer = ShardWriter(name, PAGE_FEATURES)
    for _, row in keep.iterrows():
        if writer.count >= max_pages:
            break
        text = strip_markdown(row["natural_text"])
        words = len(text.split())
        path = pdfs.get(str(row["pdf_relpath"]).replace("\\", "/")) or pdfs.get(os.path.basename(str(row["pdf_relpath"])))
        if path is None or not (5 <= words <= max_words):
            continue
        try:
            img = render_pdf_page(path, int(row["page_number"]))
        except Exception:
            continue
        writer.add({"image": image_bytes(img, "JPEG", 90), "text": text, "source": f"olmocr/{subset}", "writer": "",
                    "group": str(row["id"]), "note": str(row["url"])[:200]})
    return save(writer.finish(), name, folder)


start = step("3f. olmOCR-mix pages")
if USE_OLMOCR_EVAL and saved_dataset("olmocr_eval") is None:
    try:
        ds = olmocr_pages("00_documents", "eval", "olmocr_eval", "datasets", OLMOCR_EVAL_PAGES, OLMOCR_MAX_WORDS)
        print(f"   olmocr_eval: {len(ds):,} pages")
    except Exception as e:
        print(f"   olmOCR evaluation pages: skipped ({type(e).__name__}: {str(e)[:150]})")
if USE_OLMOCR_TRAIN and saved_dataset("olmocr_train", "raw") is None:
    try:
        ds = olmocr_pages("00_documents", "train", "olmocr_train", "raw", 10 ** 9 if not QUICK_TEST else 4, 250)
        print(f"   olmocr_train: {len(ds):,} pages")
    except Exception as e:
        print(f"   olmOCR training pages: skipped ({type(e).__name__}: {str(e)[:150]})")
note("olmocr", repo=OLMOCR_REPO, licence="ODC-BY, with Ai2's responsible-use guidelines", labels="GPT-4.1 (spot-check)")

FORM_FEATURES = Features({"image": ImageFeature(), "json": Value("string"), "kinds": Value("string"), "boxes": Value("string"),
                          "template": Value("string"), "id": Value("string"), "source": Value("string")})
# json: the answer (an object, keys in the order the instruction lists them); kinds: {key: printed | handwritten | empty |
# unknown}; boxes: {key: [x0, y0, x1, y1]} where known (template forms), else {}
FUNSD_URL = "https://guillaumejaume.github.io/FUNSD/dataset.zip"


def funsd_forms(folder, max_keys=12):
    for jpath in sorted(glob.glob(os.path.join(folder, "**", "testing_data", "annotations", "*.json"), recursive=True)):
        image = jpath.replace("annotations", "images").replace(".json", ".png")
        if not os.path.exists(image):
            continue
        entities = {e["id"]: e for e in json.load(open(jpath, encoding="utf-8"))["form"]}
        pairs = []
        for e in entities.values():
            if e.get("label") != "question":
                continue
            answers = [entities[b] for a, b in e.get("linking", []) if a == e["id"] and b in entities and entities[b].get("label") == "answer"]
            key = re.sub(r"\s+", " ", e.get("text", "")).strip().rstrip(":").strip()
            if not key or not answers:
                continue
            answers.sort(key=lambda a: (a["box"][1], a["box"][0]))
            value = re.sub(r"\s+", " ", " ".join(a.get("text", "") for a in answers)).strip()
            if value:
                pairs.append(((e["box"][1], e["box"][0]), key, value))
        pairs.sort()
        answer, seen = {}, Counter()
        for _, key, value in pairs[:max_keys]:
            seen[key] += 1
            answer[key if seen[key] == 1 else f"{key} ({seen[key]})"] = value
        if len(answer) >= 2:
            yield image, answer


start = step("3g. FUNSD")
if USE_FUNSD and saved_dataset("funsd_test") is None:
    try:
        archive = os.path.join(WORK_DIR, "funsd.zip")
        if not os.path.exists(archive):
            urllib.request.urlretrieve(FUNSD_URL, archive)
        folder = os.path.join(WORK_DIR, "funsd")
        zipfile.ZipFile(archive).extractall(folder)
        writer = ShardWriter("funsd_test", FORM_FEATURES)
        for image, answer in funsd_forms(folder):
            with Image.open(image) as img:
                writer.add({"image": image_bytes(img.convert("RGB")), "json": json.dumps(answer, ensure_ascii=False),
                            "kinds": json.dumps({k: "unknown" for k in answer}), "boxes": "{}", "template": "funsd",
                            "id": os.path.basename(image), "source": "funsd"})
        ds = save(writer.finish(), "funsd_test")
        print(f"   funsd_test: {len(ds)} forms, {sum(len(json.loads(j)) for j in ds['json'])} fields")
        note("funsd", forms=len(ds), licence="research and educational use (check the FUNSD page)")
    except Exception as e:
        print(f"   FUNSD: skipped ({type(e).__name__}: {str(e)[:150]})")

## 4. The frozen test suite (phase 0)
One list of test sets that every model and every run is scored on, written once to `eval_suite.json` and never
changed afterwards. Re-running this cell only **adds** sets whose data arrived since (the official KHATT release,
say); existing entries stay exactly as they were, so numbers from different weeks stay comparable.

Each entry records the images it scores (up to 500 per set; for the KITAB sets the first 500, as v5 did) and its
**headline subset**: the lines whose text shares no 4 words in a row with, and is not identical to, any training
line of the same family (KHATT, Muharaf, memoirs, AraMS, IAM, GNHK). Sets with no training family (HistoryAr, ADAB,
the printed KITAB sets, olmOCR pages) keep every image. The headline subset is the number to quote; the full set
is reported beside it for comparison with v5 and with published results.

| Question | Sets |
| --- | --- |
| unseen Arabic handwriting | KITAB KHATT and OnlineKhatt (headline = v5's 112 and 189 clean lines), official KHATT unique-text test (unseen writers), KHATT seen-writer test, ADAB words; Teklia KHATT test for v5 comparison (mostly seen text) |
| historical handwriting | HistoryAr (unseen source), AraMS-28k test (3 unseen books, scored letters-only), KITAB Muharaf and Muharaf test (seen source), memoir test (seen collection) |
| printed Arabic (regression guard) | PATS, SythenAR, ISI-PPT, EvArEST |
| English | IAM test (raw and with punctuation spacing normalised), GNHK test lines, olmOCR-mix pages |
| forms | held-out forms (digital, Augraphy, phone photos), FUNSD; added in section 7 |

In [ ]:
SUITE_PATH = os.path.join(DATA_DIR, "eval_suite.json")
#            name, dataset, family, question, kind, normaliser, first rows (as v5), keep only rows whose note is, tags
SUITE_SPEC = [
    ("KITAB KHATT", "kitab_KHATT", "khatt", "unseen Arabic handwriting", "line", None, True, None, ["KITAB-Bench"]),
    ("KITAB OnlineKhatt", "kitab_OnlineKhatt", "khatt", "unseen Arabic handwriting", "line", None, True, None, ["KITAB-Bench", "tablet"]),
    ("KHATT unique-text test (official)", "khatt_official_test", "khatt", "unseen Arabic handwriting", "line", None, False, "unique", ["unseen writers"]),
    ("KHATT seen-writer test (official)", "khatt_seen_writer_test", "khatt", "unseen Arabic handwriting", "line", None, False, None, ["seen writers"]),
    ("KHATT test (Teklia)", "khatt_test", "khatt", "unseen Arabic handwriting", "line", None, True, None, ["mostly seen text", "v5 comparison"]),
    ("ADAB", "kitab_ADAB", None, "unseen Arabic handwriting", "line", None, True, None, ["KITAB-Bench", "words"]),
    ("HistoryAr", "kitab_HistoryAr", None, "historical handwriting", "line", None, True, None, ["KITAB-Bench", "unseen source"]),
    ("AraMS-28k unseen books", "arams_test", "arams", "historical handwriting", "line", "letters_only", False, None, ["unseen books"]),
    ("KITAB Muharaf", "kitab_Muharaf", "muharaf", "historical handwriting", "line", None, True, None, ["KITAB-Bench", "seen source"]),
    ("Muharaf test", "muharaf_test", "muharaf", "historical handwriting", "line", None, True, None, ["seen source"]),
    ("Memoir test", "memoir_test", "memoir", "historical handwriting", "line", None, False, None, ["seen collection"]),
    ("PATS", "kitab_PATS", None, "printed Arabic", "line", None, True, None, ["KITAB-Bench"]),
    ("SythenAR", "kitab_SythenAR", None, "printed Arabic", "line", None, True, None, ["KITAB-Bench", "diacritics"]),
    ("ISI-PPT", "kitab_ISI-PPT", None, "printed Arabic", "line", None, True, None, ["KITAB-Bench", "slides"]),
    ("EvArEST", "kitab_EvArEST", None, "printed Arabic", "line", None, True, None, ["KITAB-Bench", "scene words"]),
    ("IAM test", "iam_test", "iam", "English", "line", "punct", True, None, []),
    ("GNHK test", "gnhk_test", "gnhk", "English", "line", None, False, None, ["phone photos"]),
    ("olmOCR-mix pages", "olmocr_eval", None, "English", "page", None, True, None, ["GPT-4.1 labels"]),
]
FAMILY_TRAIN = {"khatt": ["khatt_teklia_train", "khatt_official_train"], "muharaf": ["muharaf_train"], "memoir": ["memoir_train"],
                "arams": ["arams_train"], "iam": ["iam_train"], "gnhk": ["gnhk_train"]}
_family_cache = {}


def family_text(family):
    """(4-grams, plain texts) of every training line of a family."""
    if family not in _family_cache:
        grams, plains = set(), set()
        for name in FAMILY_TRAIN.get(family, []):
            ds = raw_dataset(name)
            for text in (ds["text"] if ds is not None else []):
                grams |= ngrams(text)
                plains.add(plain(text))
        _family_cache[family] = (grams, plains)
    return _family_cache[family]


def suite_entry(name, dataset, family, question, kind, normaliser, first_rows, only_note, tags):
    ds = saved_dataset(dataset)
    if ds is None or len(ds) == 0:
        return None
    rows = list(range(len(ds)))
    if only_note:
        rows = [i for i, n in enumerate(ds["note"]) if n == only_note]
    indices = rows[:SUITE_MAX] if first_rows else sorted(random.Random(0).sample(rows, min(SUITE_MAX, len(rows))))
    texts = ds.select(indices)["text"]
    if family:
        grams, plains = family_text(family)
        headline = [i for i, t in zip(indices, texts) if plain(t) not in plains and ngrams(t).isdisjoint(grams)]
    else:
        headline = list(indices)
    return {"name": name, "dataset": dataset, "rows": len(ds), "indices": indices, "headline": headline, "family": family,
            "question": question, "kind": kind, "normaliser": normaliser, "tags": tags, "created": time.strftime("%Y-%m-%d %H:%M")}


start = step("4. frozen test suite")
suite = json.load(open(SUITE_PATH)) if os.path.exists(SUITE_PATH) else {"version": "v6", "sets": []}
known = {e["name"] for e in suite["sets"]}
for spec in SUITE_SPEC:
    if spec[0] in known:
        continue
    entry = suite_entry(*spec)
    if entry is None:
        print(f"   {spec[0]}: no data yet ({spec[1]}), not in the suite")
        continue
    suite["sets"].append(entry)
    print(f"   added {entry['name']}: {len(entry['indices'])} images, headline {len(entry['headline'])}")
json.dump(suite, open(SUITE_PATH, "w"), indent=0, ensure_ascii=False)
print(f"\n{'set':36s}{'images':>8s}{'headline':>10s}  question")
for e in suite["sets"]:
    print(f"{e['name']:36s}{len(e['indices']):8d}{len(e['headline']):10d}  {e['question']}{'  [' + ', '.join(e['tags']) + ']' if e['tags'] else ''}")
note("suite", sets=len(suite["sets"]), images=sum(len(e["indices"]) for e in suite["sets"]))

## 5. Training sets: overlap filter, repeat cap, validation splits
Every training source passes three checks against **every** test set in the suite before it is saved as `train_*`:

- its text is not identical to a test line;
- it shares no 4 words in a row with a headline test line (pages from SynthDoG and olmOCR: no 8 words in a row, since
  any long English page shares some common 4-word phrase with some IAM line; the drawn sentences: no 6, as in v5);
- its image is not a near-copy (64-bit difference hash within 4 bits) of any test image, headline or not.

Then:

- **KHATT repeat cap.** At most 3 copies of each repeated passage are kept: lines are visited in a fixed random
  order and a line is kept while most of its 4-grams have been seen fewer than 3 times. The handwriting stays, the
  memorised text goes. On the official release only the fixed-text lines are capped.
- **Validation splits** for the progress checks during training (never the test sets, so choosing the best check
  cannot leak test data): official ones where the source has them (KHATT official, Muharaf, AraMS, IAM), otherwise
  carved out by page or photo (3% of memoir pages, 5% of GNHK photos, 2% of SynthDoG pages, 150 Teklia KHATT lines).

A source is filtered again automatically if the suite has changed since it was last filtered.

In [ ]:
def suite_lines():
    """Texts of every headline line, and fingerprints of every image, in the line and page sets of the suite."""
    texts, prints, cache_path = [], [], os.path.join(DATA_DIR, "suite_prints.json")
    cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
    for e in suite["sets"]:
        if e["kind"] not in ("line", "page"):
            continue
        ds = load_from_disk(dataset_path(e["dataset"]))
        texts += list(ds.select(e["headline"])["text"])
        if e["name"] not in cache:
            stored = ds.select(e["indices"]).cast_column("image", ImageFeature(decode=False))
            cache[e["name"]] = [f"{fingerprint_stored(r['image']):016x}" for r in stored]
        prints += [int(h, 16) for h in cache[e["name"]]]
    json.dump(cache, open(cache_path, "w"))
    return texts, prints


start = step("5. training sets")
TEST_TEXTS, TEST_PRINTS = suite_lines()
TEST_PLAIN = {plain(t) for t in TEST_TEXTS if plain(t)}
TEST_GRAMS = {n: set().union(*(ngrams(t, n) for t in TEST_TEXTS)) for n in (4, 6, 8)}
SUITE_HASH = hashlib.md5(json.dumps([(e["name"], e["headline"], e["indices"]) for e in suite["sets"] if e["kind"] != "form"]).encode()).hexdigest()[:12]
print(f"   test lines: {len(TEST_TEXTS):,} headline texts, {len(TEST_PRINTS):,} images; suite version {SUITE_HASH}")


def filter_training(ds, label, n=4):
    """Drop training rows that match the test suite; returns (kept rows, reasons for the dropped ones)."""
    stored = ds.cast_column("image", ImageFeature(decode=False))

    def mark(row):
        text = row["text"] or ""
        p = plain(text)
        why = ("empty" if not p else "identical to a test line" if p in TEST_PLAIN
               else f"shares {n} words in a row with a test line" if not ngrams(text, n).isdisjoint(TEST_GRAMS[n]) else "")
        return {"_why": why, "_print": f"{fingerprint_stored(row['image']):016x}"}

    marked = stored.map(mark, load_from_cache_file=False, writer_batch_size=200, desc=f"checking {label}",
                        new_fingerprint=hashlib.md5(f"{label}{time.time()}".encode()).hexdigest())
    why = list(marked["_why"])
    near = near_copy_mask([int(h, 16) for h in marked["_print"]], TEST_PRINTS)
    why = [w or ("near-copy of a test image" if c else "") for w, c in zip(why, near)]
    kept = marked.select([i for i, w in enumerate(why) if not w]).remove_columns(["_why", "_print"])
    return kept.cast_column("image", ImageFeature()), Counter(w for w in why if w)


def cap_repeats(texts, cap=KHATT_REPEAT_CAP, seed=0):
    """Indices to keep so that no 4-gram is seen more than about `cap` times."""
    order = list(range(len(texts)))
    random.Random(seed).shuffle(order)
    seen, keep = Counter(), []
    for i in order:
        grams = ngrams(texts[i]) or {plain(texts[i])}
        if sum(seen[g] >= cap for g in grams) / len(grams) < 0.5:
            keep.append(i)
            seen.update(grams)
    return sorted(keep)


def carve(ds, share, label, max_rows=None):
    """(train indices, validation indices), splitting by `group` (page, photo) so related lines stay together."""
    groups = ds["group"]
    pick = lambda i, g: int(hashlib.md5(f"{label}:{g or i}".encode()).hexdigest(), 16) % 10_000 < share * 10_000
    val = [i for i, g in enumerate(groups) if pick(i, g)]
    if max_rows is not None:
        val = val[:max_rows]
    chosen = set(val)
    return [i for i in range(len(ds)) if i not in chosen], val


def standard_lines(ds, source):
    """v5 datasets (image, text and other columns) in the common line layout."""
    if set(LINE_FEATURES) <= set(ds.column_names):
        return ds.select_columns(list(LINE_FEATURES))
    return as_lines(ds, source)


filtered_with = manifest.get("filtered_with", {})


def build_training(name, raw_name, source, n=4, val_share=None, val_max=None, cap=None, folder="raw"):
    """raw/<raw_name> -> datasets/train_<name> (+ datasets/val_<name>), unless already built for this suite version."""
    if filtered_with.get(name) == SUITE_HASH and saved_dataset(f"train_{name}") is not None:
        print(f"   train_{name}: up to date ({len(saved_dataset(f'train_{name}')):,} rows)")
        return
    ds = raw_dataset(raw_name) if folder == "raw" else saved_dataset(raw_name, folder)
    if ds is None:
        return
    t = time.time()
    ds = standard_lines(ds, source)
    kept, reasons = filter_training(ds, name, n)
    info = {"raw": len(ds), "after_filter": len(kept), "removed": dict(reasons)}
    if cap == "all":
        kept = kept.select(cap_repeats(list(kept["text"])))
    elif cap == "fixed":
        notes, texts = list(kept["note"]), list(kept["text"])
        fixed = [i for i, k in enumerate(notes) if k == "fixed"]
        keep_fixed = [fixed[j] for j in cap_repeats([texts[i] for i in fixed])]
        kept = kept.select(sorted([i for i, k in enumerate(notes) if k != "fixed"] + keep_fixed))
    if cap:
        info["after_repeat_cap"] = len(kept)
    if val_share:
        train_rows, val_rows = carve(kept, val_share, name, val_max)
        save(kept.select(val_rows), f"val_{name}")
        kept = kept.select(train_rows)
        info["validation"] = len(val_rows)
    save(kept, f"train_{name}")
    info["train"] = len(kept)
    filtered_with[name] = SUITE_HASH
    manifest["filtered_with"] = filtered_with
    note(f"train_{name}", **info)
    print(f"   train_{name}: {info['raw']:,} -> {len(kept):,} rows ({', '.join(f'{v:,} {k}' for k, v in reasons.items()) or 'nothing removed'}"
          f"{'; after the repeat cap ' + format(info.get('after_repeat_cap', 0), ',') if cap else ''}"
          f"{'; validation ' + format(info['validation'], ',') if val_share else ''}) in {(time.time() - t) / 60:.1f} min")


build_training("khatt_teklia", "khatt_teklia_train", "khatt_teklia", cap="all", val_share=0.03, val_max=150)
build_training("khatt_official", "khatt_official_train", "khatt_official", cap="fixed")
build_training("muharaf", "muharaf_train", "muharaf")
build_training("memoir", "memoir_train", "memoir", val_share=0.03)
build_training("arams", "arams_train", "arams")
build_training("iam", "iam_train", "iam")
build_training("gnhk", "gnhk_train", "gnhk", val_share=0.05)
build_training("olmocr", "olmocr_train", "olmocr", n=8)

# the drawn sentences: no 6 words in a row with any headline test line (v5 checked only against KITAB)
if filtered_with.get("text") != SUITE_HASH:
    for name in ["wikipedia_ar.json", "wikipedia_en.json", "tashkeel_ar.json"]:
        sentences = json.load(open(text_file(name), encoding="utf-8"))
        kept = [s for s in sentences if ngrams(s, 6).isdisjoint(TEST_GRAMS[6])]
        json.dump(kept, open(text_file(name), "w", encoding="utf-8"), ensure_ascii=False)
        note("text", **{f"{name}_removed_for_suite": len(sentences) - len(kept)})
        print(f"   {name}: {len(sentences) - len(kept):,} of {len(sentences):,} sentences removed (6 words in a row with a test line)")
    filtered_with["text"] = SUITE_HASH
    manifest["filtered_with"] = filtered_with
    note("text")
print(f"   ({(time.time() - start) / 60:.1f} min)")

### 5b. SynthDoG English pages, checked as they download
As in v5, with three changes: 20,000 pages instead of 2,000, up to 150 words instead of 80 (phase 4 moves English from
short lines to paragraphs), and pages saved at 1,024 px and JPEG quality 85 (about 3 GB for 20,000). Each page is checked
against the suite while it streams (no 8 words in a row with a test line, no near-copy image), so no unfiltered copy is
stored; every 50th kept page goes to the validation split. Expect about an hour: many pages are streamed past to find
ones with at most 150 words.

In [ ]:
def synthdog_pages():
    kept, started = 0, time.time()
    try:
        for row in load_dataset("naver-clova-ix/synthdog-en", split="train", streaming=True):
            text = re.sub(r"\s+", " ", json.loads(row["ground_truth"])["gt_parse"]["text_sequence"]).strip()
            if not (3 <= len(text.split()) <= SYNTHDOG_MAX_WORDS) or not ngrams(text, 8).isdisjoint(TEST_GRAMS[8]):
                continue
            img = row["image"].convert("RGB")
            if near_copy_mask([fingerprint(img)], TEST_PRINTS)[0]:
                continue
            scale = min(1.0, SYNTHDOG_MAX_SIDE / max(img.size))
            if scale < 1.0:
                img = img.resize((max(28, int(img.width * scale)), max(28, int(img.height * scale))), Image.BILINEAR)
            yield {"image": image_bytes(img, "JPEG", 85), "text": text, "source": "synthdog", "writer": "", "group": f"p{kept}", "note": ""}
            kept += 1
            if kept >= N_SYNTHDOG:
                break
            if kept % 2000 == 0:
                print(f"   {kept:,}/{N_SYNTHDOG:,} pages ({(time.time() - started) / 60:.1f} min)")
    except Exception as e:
        print(f"   SynthDoG-en stopped early ({type(e).__name__}: {str(e)[:100]})")


if USE_SYNTHDOG and saved_dataset("train_synthdog") is None:
    start = time.time()
    train_pages, val_pages = ShardWriter("train_synthdog", PAGE_FEATURES), ShardWriter("val_synthdog", PAGE_FEATURES)
    for k, row in enumerate(synthdog_pages()):
        (val_pages if k % 50 == 49 else train_pages).add(row)
    val_count = len(save(val_pages.finish(), "val_synthdog"))
    ds = save(train_pages.finish(), "train_synthdog")
    filtered_with["synthdog"] = SUITE_HASH
    manifest["filtered_with"] = filtered_with
    note("train_synthdog", train=len(ds), validation=val_count, max_words=SYNTHDOG_MAX_WORDS,
         licence="not stated on the card; generator MIT, text Wikipedia CC-BY-SA")
    print(f"   train_synthdog: {len(ds):,} pages, val_synthdog {val_count:,} ({(time.time() - start) / 60:.1f} min)")
elif USE_SYNTHDOG:
    print(f"   train_synthdog: already saved ({len(saved_dataset('train_synthdog')):,} pages)")

## 6. Writer audit
The roadmap's ground rule is that handwriting test sets keep writers apart from training, **verified from writer
IDs rather than assumed**. Our Teklia copies of KHATT and IAM carry no writer IDs, so they are matched to the
official releases here (KHATT: same text and a near-identical image; IAM: same text through IAM's `lines.txt`).

Without the official KHATT release the audit can still answer one question from the text alone: how much of our
Teklia KHATT copy is repeated passages, and how much of its test text also appears in training.

In [ ]:
start = step("6. writer audit")
audit = {}
teklia_train, teklia_test = raw_dataset("khatt_teklia_train"), saved_dataset("khatt_test")
if teklia_train is not None and teklia_test is not None:
    train_texts, test_texts = list(teklia_train["text"]), list(teklia_test["text"])
    grams_train = set().union(*(ngrams(t) for t in train_texts))
    count_of = Counter(g for t in train_texts for g in ngrams(t))
    repeated = sum(1 for t in train_texts if ngrams(t) and sum(count_of[g] >= 6 for g in ngrams(t)) / len(ngrams(t)) > 0.5)
    shared = sum(1 for t in test_texts if not ngrams(t).isdisjoint(grams_train))
    audit["teklia_text"] = {"train_lines": len(train_texts), "train_lines_mostly_repeated": repeated,
                            "test_lines": len(test_texts), "test_lines_sharing_a_4gram_with_train": shared}
    print(f"   Teklia KHATT, from the text alone: {repeated:,} of {len(train_texts):,} training lines are mostly passages that "
          f"5+ other lines repeat; {shared:,} of {len(test_texts):,} test lines share 4 words in a row with training")
official = {s: (raw_dataset(n) if f == "raw" else saved_dataset(n, f)) for s, n, f in [("train", "khatt_official_train", "raw"), ("validation", "khatt_official_validation", "datasets"),
                                                    ("test", "khatt_official_test", "datasets"), ("seen", "khatt_seen_writer_test", "datasets")]}
if teklia_train is not None and official["train"] is not None:
    # index the official lines by plain text, with image fingerprints to break ties between writers who copied the same text
    index = defaultdict(list)
    for split, ds in official.items():
        if ds is None:
            continue
        stored = ds.cast_column("image", ImageFeature(decode=False))
        for r in stored:
            index[plain(r["text"])].append((split, r["writer"], r["note"], fingerprint_stored(r["image"])))

    def match(ds):
        found = []
        for r in ds.cast_column("image", ImageFeature(decode=False)):
            candidates = index.get(plain(r["text"]), [])
            if not candidates:
                found.append(None)
                continue
            bits = fingerprint_stored(r["image"])
            best = min(candidates, key=lambda c: bin(c[3] ^ bits).count("1"))
            found.append(best if bin(best[3] ^ bits).count("1") <= 8 else None)
        return found

    m_train, m_test = match(teklia_train), match(teklia_test)
    train_writers = {m[1] for m in m_train if m}
    test_matched = [m for m in m_test if m]
    audit["teklia_vs_official"] = {
        "train_matched": sum(m is not None for m in m_train), "test_matched": len(test_matched),
        "train_kinds": dict(Counter(m[2] for m in m_train if m)), "test_kinds": dict(Counter(m[2] for m in test_matched)),
        "test_lines_whose_writer_is_in_teklia_train": sum(m[1] in train_writers for m in test_matched),
        "teklia_test_lines_in_official_train": sum(m[0] == "train" for m in test_matched)}
    print("   Teklia KHATT matched to the official release:", json.dumps(audit["teklia_vs_official"], ensure_ascii=False))
    print("   The official release is what phase 2 trains on; the Teklia copy stays only as a v5 comparison.")
elif teklia_train is not None:
    print("   no official KHATT release yet: writer overlap in the Teklia copy cannot be checked (see section 3b)")
lines_txt = glob.glob(os.path.join(IAM_OFFICIAL_DIR, "**", "lines.txt"), recursive=True)
forms_txt = glob.glob(os.path.join(IAM_OFFICIAL_DIR, "**", "forms.txt"), recursive=True)
if lines_txt and forms_txt:
    writer_of_form = {}
    for line in open(forms_txt[0], encoding="utf-8", errors="replace"):
        if not line.startswith("#") and len(line.split()) > 1:
            writer_of_form[line.split()[0]] = line.split()[1]
    writers_by_text = defaultdict(set)
    for line in open(lines_txt[0], encoding="utf-8", errors="replace"):
        parts = line.strip().split(" ")
        if line.startswith("#") or len(parts) < 9:
            continue
        form = "-".join(parts[0].split("-")[:2])
        writers_by_text[plain(" ".join(parts[8:]).replace("|", " "))].add(writer_of_form.get(form, "?"))
    iam_train, iam_test = raw_dataset("iam_train"), saved_dataset("iam_test")
    train_writers = set().union(*(writers_by_text.get(plain(t), set()) for t in iam_train["text"]))
    test_sets = [writers_by_text.get(plain(t), set()) for t in iam_test["text"]]
    audit["iam"] = {"test_matched": sum(bool(s) for s in test_sets),
                    "test_lines_with_a_writer_in_train": sum(bool(s & train_writers) for s in test_sets)}
    print("   IAM matched to the official lines.txt:", audit["iam"])
note("writer_audit", **audit)
print(f"   ({time.time() - start:.0f} s)")

## 7. Forms: templates, the held-out form set, phone photos, training fills
The held-out form set measures what v5 could not: forms whose **layout, writers, fonts and capture** were never seen
in training. One person can build it without reading Arabic, because code writes every label. The steps, and what
you do in each:

| Step | You | This notebook |
| --- | --- | --- |
| 1. collect | download 40–60 **blank** forms (PDF or image) from public sites, checking each site's terms; Arabic and bilingual (bank, telecom, school, rental, service applications). Never filled forms, never identity documents. Put them in `MyDrive/arabic_vlm/form_templates/` | turns each into a 1,240 px wide page and finds the boxes, underlines and character combs where values go |
| 2. mark | open `form_templates/overlays/<template>.png` (numbered red boxes) and `form_templates/fields/<template>.json`; type a key (from the list printed below) into `"key"` for each value box, leave the others empty. You can add a box by copying an entry and changing its pixel coordinates (the overlay has a 100 px grid) | optional: reads the label beside each box and suggests a key (`SUGGEST_KEYS`, needs the v5 checkpoint; faster on a GPU) |
| 3. fill | – | 15 templates are held out once, at random. Each is filled 10 times with fictional values: typed in fonts no model trains on, or handwritten, pasted from the test lines of writers no model trains on (KHATT test, IAM test, GNHK test), so every JSON label is exact. A QR code with the form's ID goes in a free corner |
| 4. capture | print `forms/heldout/print/*.pdf` (A4), photograph each sheet with a phone (different light, angles, a fold or two), put the photos in `MyDrive/arabic_vlm/form_photos/` and run this section again | reads each photo's QR code and pairs it with its label; also makes a print-scan-fax copy of every form with Augraphy |
| 5. train templates | – | the other templates are filled 2,000 times for training, with training fonts and training writers only |

The held-out templates, the training templates, the fonts and the handwriting writers are disjoint by construction,
so a model cannot score well on the held-out set by having memorised any of them.

In [ ]:
import cv2
import qrcode

TEMPLATE_WIDTH = 1240              # A4 at 150 dpi
N_HELDOUT_TEMPLATES = 15
FILLS_PER_HELDOUT = 10
AUGRAPHY_COPIES = 1
N_TRAIN_FILLS = 2000
SUGGEST_KEYS = False               # True: read the printed label beside each box with the v5 model and suggest a key
P_EMPTY = 0.1                      # share of fields left empty (their JSON value is null)
if QUICK_TEST:
    N_HELDOUT_TEMPLATES, FILLS_PER_HELDOUT, N_TRAIN_FILLS = 1, 2, 4
FIELDS_DIR = os.path.join(FORM_TEMPLATES_DIR, "fields")
OVERLAY_DIR = os.path.join(FORM_TEMPLATES_DIR, "overlays")
HELDOUT_DIR = os.path.join(DATA_DIR, "forms", "heldout")
TEMPLATE_EXTS = (".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".webp")
VOCAB = json.load(open(text_file("vocab.json"), encoding="utf-8"))
FIELD_KIND = {k: v["kind"] for k, v in VOCAB["field_types"].items()}

# --- drawing text (the same rules as the training notebook: raqm lays out Arabic, digit groups stay in reading order)
HAS_RAQM = features.check("raqm")
_fonts = {}


def get_font(name_or_path, size):
    path = name_or_path if os.path.isabs(name_or_path) or os.path.exists(name_or_path) else os.path.join(FONT_DIR, name_or_path)
    if (path, size) not in _fonts:
        _fonts[(path, size)] = ImageFont.truetype(path, size, layout_engine=ImageFont.Layout.RAQM if HAS_RAQM else ImageFont.Layout.BASIC)
    return _fonts[(path, size)]


def is_arabic(text):
    return re.search(r"[؀-ٟ٪-ۿ]", text) is not None


def arabic_font_needed(text):
    """Arabic letters or Arabic-Indic digits: Latin fonts have neither, so such text is drawn in an Arabic font."""
    return is_arabic(text) or re.search(r"[٠-٩]", text) is not None


def prepare(text):
    if HAS_RAQM:
        return text, ({"direction": "rtl", "language": "ar"} if is_arabic(text) else {"direction": "ltr"})
    if is_arabic(text):
        import arabic_reshaper
        from bidi.algorithm import get_display
        return get_display(arabic_reshaper.reshape(text)), {}
    return text, {}


def typeset_ltr(text):
    """A left-to-right mark before each separator keeps Arabic-Indic digit groups in reading order (see the training
    notebook, section E). The marks are invisible and never part of the answer."""
    if re.search(r"[٠-٩]", text) and not is_arabic(text):
        return re.sub(r"(?<=[٠-٩])(?=[\s\-/.:])", "‎", text)
    return text


def text_width(draw, text, font):
    shown, options = prepare(text)
    return draw.textlength(shown, font=font, **options)


def draw_text_at(draw, x, y, text, font, colour):
    shown, options = prepare(text)
    draw.text((x, y), shown, font=font, fill=colour, **options)


# --- fictional values (the same generator as the training notebook's form generator v2) ----------------------------
TO_ARABIC_DIGITS = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")


def person(rng):
    first = rng.choice(VOCAB["arabic_female"] if rng.random() < 0.5 else VOCAB["arabic_male"])
    father, family = rng.choice(VOCAB["arabic_male"]), rng.choice(VOCAB["arabic_family"])
    return f"{first[0]} {father[0]} {family[0]}", f"{first[1]} {father[1]} {family[1]}"


def english_person(rng):
    return f"{rng.choice(VOCAB['english_first'])} {rng.choice(VOCAB['english_last'])}"


def a_date(rng, years):
    year, month, day = rng.randint(*years), rng.randint(1, 12), rng.randint(1, 28)
    iso = f"{year:04d}-{month:02d}-{day:02d}"
    shown = rng.choice([f"{day:02d}/{month:02d}/{year}", f"{day:02d}-{month:02d}-{year}", f"{day:02d}.{month:02d}.{year}", iso,
                        f"{year}/{month:02d}/{day:02d}", f"{day} {VOCAB['months_en'][month - 1]} {year}",
                        f"{day} {VOCAB['months_ar'][month - 1]} {year}"])
    return iso, shown


def make_value(kind, rng, script="ar", sentences=None):
    """(answer, text drawn on the form). Answers follow the form instruction: Western digits, dates as YYYY-MM-DD,
    phone numbers as digits only; everything else as written."""
    show = (lambda s: s.translate(TO_ARABIC_DIGITS)) if (script == "ar" and rng.random() < 0.4) else (lambda s: s)
    pick = lambda pool: rng.choice(VOCAB[pool])[0 if script == "ar" else 1]
    if kind == "name_ar":
        value = person(rng)[0] if script == "ar" else english_person(rng)
        return value, value
    if kind == "name_en":
        value = person(rng)[1] if rng.random() < 0.6 else english_person(rng)
        return value, value
    if kind in ("date", "dob", "past_date", "future_date"):
        years = {"date": (2020, 2027), "dob": (1950, 2008), "past_date": (2015, 2026), "future_date": (2026, 2034)}[kind]
        iso, shown = a_date(rng, years)
        return iso, shown if re.search(r"[A-Za-z]", shown) else show(shown)            # no Arabic-Indic digits beside an English month
    if kind in ("phone", "mobile"):
        digits = ("05" + str(rng.choice([0, 2, 4, 5, 6, 8])) if kind == "mobile" else "0" + str(rng.choice([2, 3, 4, 6, 7, 9])))
        digits += "".join(str(rng.randint(0, 9)) for _ in range(7))
        spaced = f"{digits[:3]} {digits[3:6]} {digits[6:]}" if kind == "mobile" else f"{digits[:2]} {digits[2:5]} {digits[5:]}"
        shown = rng.choice([digits, spaced, "+971 " + spaced[1:], f"{digits[:3]}-{digits[3:]}"])
        return digits, show(shown) if rng.random() < 0.3 else shown
    if kind == "email":
        parts = (person(rng)[1] if rng.random() < 0.6 else english_person(rng)).lower().replace("al ", "al").split()
        value = rng.choice([f"{parts[0]}.{parts[-1]}", f"{parts[0][0]}{parts[-1]}", f"{parts[0]}{rng.randint(1, 99)}"]) + "@" + rng.choice(VOCAB["email_domains"])
        return value, value
    if kind in ("po_box", "quantity"):
        number = str(rng.randint(100, 999999) if kind == "po_box" else rng.randint(1, 500))
        return number, show(number)
    if kind in ("emirate", "city", "area", "nationality", "gender", "marital", "occupation", "relationship", "grade"):
        pool = {"emirate": "emirates", "area": "areas", "nationality": "nationalities", "marital": "marital",
                "occupation": "occupations", "relationship": "relationships", "grade": "grades", "gender": "gender"}.get(kind)
        value = pick(pool) if pool else rng.choice(VOCAB["emirates"] + VOCAB["cities"])[0 if script == "ar" else 1]
        return value, value
    if kind == "address":
        (ea, ee), (sa, se), n = rng.choice(VOCAB["emirates"]), rng.choice(VOCAB["streets"]), rng.randint(1, 120)
        value = f"{ea}، {sa}، مبنى {n}" if script == "ar" else f"Building {n}, {se}, {ee}"
        return value, show(value)
    if kind == "company":
        (wa, we), (ka, ke) = rng.choice(VOCAB["company_words"]), rng.choice(VOCAB["company_kinds"])
        value = f"شركة {wa} {ka}" if script == "ar" else f"{we} {ke}"
        return value, value
    if kind == "school":
        wa, we = rng.choice(VOCAB["school_words"])
        value = f"مدرسة {wa}" if script == "ar" else f"{we} School"
        return value, value
    if kind in ("reference", "invoice", "licence"):
        prefix = rng.choice({"reference": ["SR", "REQ", "REF", "APP"], "invoice": ["INV", "TAX-INV", "IN"], "licence": ["TL", "CN", "LIC"]}[kind])
        number = str(rng.randint(10_000, 9_999_999))
        return f"{prefix}-{number}", f"{prefix}-{show(number)}"
    if kind == "emirates_id":                       # a made-up number in the usual layout, never a real one
        value = f"784-{rng.randint(1950, 2010)}-{rng.randint(1_000_000, 9_999_999)}-{rng.randint(0, 9)}"
        return value, value
    if kind == "passport":
        value = rng.choice("ABCDEFGHJKLMNPRSTUVWXYZ") + str(rng.randint(1_000_000, 99_999_999))
        return value, value
    if kind in ("amount", "amount_small"):
        amount = (rng.randint(50, 50_000) if kind == "amount" else rng.randint(1, 2_500)) + rng.choice([0, 0.25, 0.5, 0.75])
        shown = rng.choice([f"{amount:,.2f}", f"AED {amount:,.2f}", f"{amount:,.2f} درهم", f"{amount:.2f}"])
        return f"{amount:.2f}", show(shown)
    if kind == "plate":
        value = f"{rng.choice('ABCDEFGHIJKLMNOPQRS')} {rng.randint(1, 99_999)}"
        return value, value
    if kind == "choice":
        value = rng.choice(rng.choice(VOCAB["choices"]))[0 if script == "ar" else 1]
        return value, value
    # free text: a few words of a held-out or training sentence
    words = rng.choice(sentences or ["ملاحظات عامة حول الطلب"]).split()
    start = rng.randint(0, max(0, len(words) - 4))
    value = " ".join(words[start:start + rng.randint(3, 8)])
    return value, show(value)


def split_pool(sentences):
    """The training notebook's split: the same shuffle, the first 5% held out."""
    sentences = list(sentences)
    random.Random(0).shuffle(sentences)
    cut = max(1, len(sentences) // 20)
    return sentences[cut:] or sentences, sentences[:cut]


SENTENCES = {"ar": split_pool(json.load(open(text_file("wikipedia_ar.json"), encoding="utf-8"))),
             "en": split_pool(json.load(open(text_file("wikipedia_en.json"), encoding="utf-8")))}
print("form keys you can use in fields/*.json:\n   " + ", ".join(VOCAB["field_types"]))
print("   (any other key works too: its value is then treated as free text)")

### 7a. Templates: pages, candidate boxes, overlays and the `fields` files

In [ ]:
def template_files():
    if not os.path.isdir(FORM_TEMPLATES_DIR):
        return []
    return sorted(p for p in glob.glob(os.path.join(FORM_TEMPLATES_DIR, "*")) if p.lower().endswith(TEMPLATE_EXTS))


def template_id(path):
    name = os.path.basename(path)
    slug = re.sub(r"[^A-Za-z0-9]+", "_", os.path.splitext(name)[0]).strip("_")[:40]
    return f"{slug or 'form'}_{hashlib.md5(name.encode()).hexdigest()[:6]}"


def rasterize(path, page=0):
    if path.lower().endswith(".pdf"):
        import pypdfium2 as pdfium
        pdf = pdfium.PdfDocument(path)
        p = pdf[min(page, len(pdf) - 1)]
        img = p.render(scale=TEMPLATE_WIDTH / p.get_size()[0]).to_pil().convert("RGB")
        pdf.close()
        return img
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    return img.resize((TEMPLATE_WIDTH, max(1, int(img.height * TEMPLATE_WIDTH / img.width))), Image.LANCZOS)


def overlap(a, b):
    """Share of box a covered by box b."""
    w = min(a[2], b[2]) - max(a[0], b[0])
    h = min(a[3], b[3]) - max(a[1], b[1])
    return max(0, w) * max(0, h) / max(1, (a[2] - a[0]) * (a[3] - a[1]))


def detect_candidates(img):
    """Boxes where a value could be written: closed boxes and table cells, rows of character cells (combs), and
    underlines (solid, dashed or dotted) with the space above them. Sorted top to bottom, then right to left."""
    g = np.asarray(img.convert("L"))
    H, W = g.shape
    ink = ((g < 170) * 255).astype(np.uint8)
    joined = cv2.morphologyEx(ink, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (max(5, W // 150), 1)))
    horizontal = cv2.morphologyEx(joined, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (max(30, W // 20), 1)))
    vertical = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(15, H // 70))))
    rules = cv2.dilate(cv2.bitwise_or(horizontal, vertical), np.ones((3, 3), np.uint8))
    found, cells = [], []
    n, _, stats, _ = cv2.connectedComponentsWithStats(255 - rules, connectivity=4)
    for i in range(1, n):
        x, y, w, h, area = (int(v) for v in stats[i])
        if x == 0 or y == 0 or x + w >= W or y + h >= H or area < 0.6 * w * h or not (H * 0.01 <= h <= H * 0.12):
            continue
        if w < W * 0.04:
            if 0.5 < w / max(1, h) < 2.0:
                cells.append((x, y, x + w, y + h))
        elif w <= W * 0.95:
            found.append(("box", x, y, x + w, y + h))
    rows = []
    for c in sorted(cells, key=lambda c: (round(c[1] / 8), c[0])):
        for row in rows:
            last = row[-1]
            if abs(c[1] - last[1]) < (last[3] - last[1]) / 2 and 0 <= c[0] - last[2] < (last[2] - last[0]):
                row.append(c)
                break
        else:
            rows.append([c])
    for row in rows:
        if len(row) >= 4:
            found.append(("comb", row[0][0], min(c[1] for c in row), row[-1][2], max(c[3] for c in row)))
    n, _, stats, _ = cv2.connectedComponentsWithStats(horizontal, connectivity=8)
    text_h = int(H * 0.022)
    for i in range(1, n):
        x, y, w, h, _ = (int(v) for v in stats[i])
        if w < W * 0.06 or w > W * 0.95 or h > max(6, H * 0.006):
            continue
        box = (x, max(0, y - int(text_h * 1.5)), x + w, y + h)
        is_edge = any((abs(y - b[2]) <= 8 or abs(y - b[4]) <= 8) and min(x + w, b[3]) - max(x, b[1]) > 0.5 * w for b in found)
        if not is_edge and not any(overlap(box, b[1:]) > 0.3 for b in found):     # not the top or bottom edge of a box
            found.append(("line", *box))
    found.sort(key=lambda b: (round(b[2] / (H * 0.015)), -b[3]))
    return [{"id": i + 1, "box": [int(v) for v in b[1:]], "style": b[0], "key": ""} for i, b in enumerate(found)]


def overlay(img, fields):
    out = img.copy()
    d = ImageDraw.Draw(out)
    small = get_font("NotoSans.ttf", 14) if os.path.exists(os.path.join(FONT_DIR, "NotoSans.ttf")) else ImageFont.load_default()
    label = get_font("NotoSans.ttf", 20) if os.path.exists(os.path.join(FONT_DIR, "NotoSans.ttf")) else ImageFont.load_default()
    for x in range(0, out.width, 100):
        d.line([(x, 0), (x, out.height)], fill=(205, 222, 255))
        d.text((x + 2, 2), str(x), fill=(90, 120, 200), font=small)
    for y in range(100, out.height, 100):
        d.line([(0, y), (out.width, y)], fill=(205, 222, 255))
        d.text((2, y + 2), str(y), fill=(90, 120, 200), font=small)
    for f in fields:
        colour = (0, 150, 60) if f.get("key") else (220, 30, 30)
        d.rectangle(f["box"], outline=colour, width=3)
        d.text((f["box"][0] + 3, max(0, f["box"][1] - 22)), f"{f['id']}" + (f" {f['key']}" if f.get("key") else ""), fill=colour, font=label)
    return out


start = step("7a. form templates")
if os.path.isdir(FORM_TEMPLATES_DIR):
    os.makedirs(FIELDS_DIR, exist_ok=True)
TEMPLATES = {}                    # id -> {"path", "spec", "keyed"}
for path in template_files():
    tid = template_id(path)
    spec_path = os.path.join(FIELDS_DIR, f"{tid}.json")
    spec = json.load(open(spec_path, encoding="utf-8")) if os.path.exists(spec_path) else None
    img = rasterize(path, spec.get("page", 0) if spec else 0)
    if spec is None:
        spec = {"template": os.path.basename(path), "page": 0, "width": img.width, "height": img.height, "language": "bilingual",
                "help": "Type a key into 'key' for every box that holds a value (see the list in the notebook); leave '' to ignore "
                        "a box. 'language': 'ar', 'en' or 'bilingual'. To add a box, copy an entry and change its pixel box "
                        "[left, top, right, bottom].",
                "fields": detect_candidates(img)}
        json.dump(spec, open(spec_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    os.makedirs(OVERLAY_DIR, exist_ok=True)
    overlay(img, spec["fields"]).save(os.path.join(OVERLAY_DIR, f"{tid}.png"))
    TEMPLATES[tid] = {"path": path, "spec": spec, "keyed": [f for f in spec["fields"] if f.get("key")]}
ready = [t for t, v in TEMPLATES.items() if v["keyed"]]
print(f"   {len(TEMPLATES)} templates in {FORM_TEMPLATES_DIR}; {len(ready)} have keys marked"
      + ("" if TEMPLATES else " (none yet: see step 1 above)"))
for tid, v in list(TEMPLATES.items())[:60]:
    print(f"      {tid}: {len(v['spec']['fields'])} candidate boxes, {len(v['keyed'])} keyed")

# which templates are held out: chosen once, at random, when there are enough of them; later additions go to training
SPLIT_PATH = os.path.join(FORM_TEMPLATES_DIR, "split.json")
split = json.load(open(SPLIT_PATH)) if os.path.exists(SPLIT_PATH) else None
if split is None and len(TEMPLATES) >= N_HELDOUT_TEMPLATES + (5 if not QUICK_TEST else 0) and TEMPLATES:
    ids = sorted(TEMPLATES)
    random.Random(2026).shuffle(ids)
    split = {"heldout": sorted(ids[:N_HELDOUT_TEMPLATES]), "train": sorted(ids[N_HELDOUT_TEMPLATES:]), "created": time.strftime("%Y-%m-%d")}
    json.dump(split, open(SPLIT_PATH, "w"), indent=1)
    print(f"   held out {N_HELDOUT_TEMPLATES} templates for testing (saved in {SPLIT_PATH}; never changed afterwards)")
elif split is None and TEMPLATES:
    print(f"   the held-out templates are chosen once there are at least {N_HELDOUT_TEMPLATES + 5} templates")
if split:
    split["train"] = sorted(set(split["train"]) | (set(TEMPLATES) - set(split["heldout"])))
    json.dump(split, open(SPLIT_PATH, "w"), indent=1)
note("form_templates", templates=len(TEMPLATES), keyed=len(ready), heldout=len(split["heldout"]) if split else 0)

### 7b. Optional: suggest keys from the printed labels
For Arabic-only forms: reads the text to the right and to the left of each unmarked box with the v5 stage-3 model,
matches it against the Arabic and English labels of the 40 field types, and writes `label_read` and `suggested_key`
into the fields file. Copy a suggestion into `key` if it is right (or set `AUTO_ACCEPT = True` to copy every
suggestion; check the overlay afterwards). For a label it cannot match, paste `label_read` into a translator.

In [ ]:
AUTO_ACCEPT = False
LABEL_TO_KEY = {plain(label): key for key, v in VOCAB["field_types"].items() for label in v["ar"] + v["en"]}
LABEL_TO_KEY.update({plain(k): v for k, v in {"الاسم": "full_name", "اسم": "full_name", "name": "full_name", "tel": "phone", "telephone": "phone",
                                               "هاتف": "phone", "جوال": "mobile", "mobile": "mobile", "e mail": "email", "email": "email",
                                               "توقيع": "", "signature": "", "التوقيع": ""}.items()})


def suggest_key(text):
    p = plain(text)
    best = max((label for label in LABEL_TO_KEY if label and label in p), key=len, default=None)
    return LABEL_TO_KEY[best] if best is not None else ""


if SUGGEST_KEYS and TEMPLATES:
    import torch
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    device = "cuda" if torch.cuda.is_available() else "cpu"
    reader = Qwen2_5_VLForConditionalGeneration.from_pretrained(V5_CHECKPOINT, dtype=torch.bfloat16 if device == "cuda" else torch.float32).to(device).eval()
    reader_processor = AutoProcessor.from_pretrained(V5_CHECKPOINT)
    reader_processor.tokenizer.padding_side = "left"

    @torch.no_grad()
    def read_crops(crops):
        prompt = reader_processor.apply_chat_template([{"role": "user", "content": [{"type": "image"}, {"type": "text", "text":
                 "Read all the text in the image exactly as written. اقرأ النص كما هو مكتوب."}]}], tokenize=False, add_generation_prompt=True)
        out = []
        for i in range(0, len(crops), 4):
            inputs = reader_processor(text=[prompt] * len(crops[i:i + 4]), images=crops[i:i + 4], padding=True, return_tensors="pt").to(device)
            new = reader.generate(**inputs, max_new_tokens=24, do_sample=False)[:, inputs["input_ids"].shape[1]:]
            out += [t.strip() for t in reader_processor.batch_decode(new, skip_special_tokens=True)]
        return out

    for tid, v in TEMPLATES.items():
        spec, todo = v["spec"], [f for f in v["spec"]["fields"] if not f.get("key") and "label_read" not in f]
        if not todo:
            continue
        img = rasterize(v["path"], spec.get("page", 0))
        crops = []
        for f in todo:
            x0, y0, x1, y1 = f["box"]
            pad = max(6, (y1 - y0) // 3)
            crops.append(img.crop((min(img.width - 2, x1 + 4), max(0, y0 - pad), min(img.width, x1 + int(0.35 * img.width)), min(img.height, y1 + pad))))
            crops.append(img.crop((max(0, x0 - int(0.35 * img.width)), max(0, y0 - pad), max(2, x0 - 4), min(img.height, y1 + pad))))
        reads = read_crops(crops)
        for j, f in enumerate(todo):
            right, left = reads[2 * j], reads[2 * j + 1]
            f["label_read"] = f"right: {right} | left: {left}"
            f["suggested_key"] = suggest_key(right) or suggest_key(left)
            if AUTO_ACCEPT and f["suggested_key"]:
                f["key"] = f["suggested_key"]
        json.dump(spec, open(os.path.join(FIELDS_DIR, f"{tid}.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        print(f"   {tid}: {sum(bool(f['suggested_key']) for f in todo)} of {len(todo)} boxes got a suggestion")
    del reader
else:
    print("   skipped (SUGGEST_KEYS = False)" if TEMPLATES else "   no templates yet")

### 7c. The held-out form set, its printouts and its Augraphy copies

In [ ]:
FONT_LIST = json.load(open(os.path.join(FONT_DIR, "fonts.json")))
HELD_FONTS = set(FONT_LIST["heldout_forms"])
SPECIAL = set(FONT_LIST["handwriting_like"]) | set(FONT_LIST["display"]) | set(FONT_LIST["mono"])
FORM_FONTS = {   # (typed Arabic, handwritten Arabic, typed Latin, handwritten Latin, mono) for each side of the split
    "heldout": {"ar": [f for f in ["Fustat.ttf"] if f in HELD_FONTS] or ["NotoNaskhArabic.ttf"],
                "ar_hand": [f for f in ["Alkalami.ttf"] if f in HELD_FONTS] or ["ArefRuqaa.ttf"],
                "en": [f for f in ["PublicSans.ttf"] if f in HELD_FONTS] or ["NotoSans.ttf"],
                "en_hand": [f for f in ["ReenieBeanie.ttf", "Handlee.ttf"] if f in HELD_FONTS] or ["Caveat.ttf"],
                "mono": [f for f in ["SpaceMono.ttf"] if f in HELD_FONTS] or ["CourierPrime.ttf"]},
    "train": {"ar": [f for f in FONT_LIST["arabic"] if f not in SPECIAL | HELD_FONTS],
              "ar_hand": [f for f in FONT_LIST["handwriting_like"] if f in FONT_LIST["arabic"] and f not in HELD_FONTS],
              "en": [f for f in FONT_LIST["english"] if f not in SPECIAL | HELD_FONTS],
              "en_hand": [f for f in FONT_LIST["handwriting_like"] if f in FONT_LIST["english"] and f not in HELD_FONTS],
              "mono": [f for f in FONT_LIST["mono"] if f not in HELD_FONTS]},
}


class CropPool:
    """Short handwritten lines (at most 7 words) from the given datasets (only the listed rows, if given), read lazily
    when a form needs one."""

    def __init__(self, names):
        self.items = []
        for name, folder, rows in names:
            ds = saved_dataset(name, folder)
            if ds is None:
                continue
            texts = ds["text"]
            for i in (range(len(ds)) if rows is None else rows):
                if texts[i] and 1 <= len(texts[i].split()) <= 7:
                    self.items.append((ds, i))

    def __len__(self):
        return len(self.items)

    def sample(self, rng):
        ds, i = rng.choice(self.items)
        row = ds[i]
        return row["image"], re.sub(r"\s+", " ", row["text"]).strip()


def headline_rows(name):
    """The headline (text-disjoint) rows of a suite set, or None if the set is not in the suite."""
    entry = next((e for e in suite["sets"] if e["name"] == name), None)
    return entry["headline"] if entry else None


CROPS = {   # handwriting for the held-out forms comes only from test lines of writers no model trains on, text-disjoint ones
    "heldout": {"ar": CropPool([("khatt_official_test", "datasets", headline_rows("KHATT unique-text test (official)"))])
                if saved_dataset("khatt_official_test") is not None
                else CropPool([("kitab_KHATT", "datasets", headline_rows("KITAB KHATT")), ("kitab_OnlineKhatt", "datasets", headline_rows("KITAB OnlineKhatt"))]),
                "en": CropPool([("iam_test", "datasets", headline_rows("IAM test")), ("gnhk_test", "datasets", headline_rows("GNHK test"))])},
    "train": {"ar": CropPool([("train_khatt_official", "datasets", None), ("train_khatt_teklia", "datasets", None), ("train_memoir", "datasets", None)]),
              "en": CropPool([("train_iam", "datasets", None), ("train_gnhk", "datasets", None)])},
}
print("   handwriting for forms:", {side: {k: len(v) for k, v in pools.items()} for side, pools in CROPS.items()})


def ink(rng, hand):
    if hand and rng.random() < 0.6:
        return (rng.randint(10, 40), rng.randint(20, 60), rng.randint(110, 170))      # blue pen
    return tuple([rng.randint(0, 45)] * 3)


def draw_value(img, box, text, font_name, hand, rng, style):
    x0, y0, x1, y1 = box
    draw = ImageDraw.Draw(img)
    size = max(12, min(44, int((y1 - y0) * (0.7 if style == "line" else 0.6))))
    font = get_font(font_name, size)
    shown = typeset_ltr(text)
    while text_width(draw, shown, font) > (x1 - x0) - 12 and size > 12:
        size -= 2
        font = get_font(font_name, size)
    width = text_width(draw, shown, font)
    shift = rng.uniform(0, 0.25) * max(0, (x1 - x0) - width - 12) if hand else 0     # handwriting starts a little in
    x = (x1 - 6 - width - shift) if arabic_font_needed(text) else (x0 + 6 + shift)
    x = min(max(x, x0), max(x0, x1 - width))                                          # never outside its box
    y = (y1 - size * 1.35) if style == "line" else (y0 + ((y1 - y0) - size * 1.3) / 2)
    if hand:
        y += rng.uniform(-0.08, 0.08) * size
    draw_text_at(draw, x, y, shown, font, ink(rng, hand))


def paste_crop(img, box, crop, right_aligned, rng):
    x0, y0, x1, y1 = box
    crop = crop.convert("L")
    a = np.asarray(crop, dtype=np.float32)
    lo, hi = np.percentile(a, 1), np.percentile(a, 90)
    if hi - lo > 30:                                   # stretch the contrast, then paper to pure white: only the ink lands on the form
        a = np.clip((a - lo) / (hi - lo) * 255, 0, 255)
    a = a.astype(np.uint8)
    _, paper = cv2.threshold(a, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    crop = Image.fromarray(np.where(paper > 0, 255, a).astype(np.uint8))
    scale = min((y1 - y0 - 4) / crop.height, (x1 - x0 - 10) / crop.width, 1.5)
    if scale <= 0:
        return False
    crop = crop.resize((max(1, int(crop.width * scale)), max(1, int(crop.height * scale))), Image.LANCZOS)
    px = int(x1 - 5 - crop.width) if right_aligned else int(x0 + 5)
    py = int(y0 + (y1 - y0 - crop.height) / 2)
    px, py = max(0, min(px, img.width - crop.width)), max(0, min(py, img.height - crop.height))
    region = img.crop((px, py, px + crop.width, py + crop.height))
    pen = Image.new("RGB", crop.size, ink(rng, True))
    img.paste(Image.composite(pen, region, ImageOps.invert(crop)), (px, py))
    return True


def add_qr(img, text, boxes):
    code = qrcode.QRCode(box_size=4, border=2, error_correction=qrcode.constants.ERROR_CORRECT_M)
    code.add_data(text)
    code.make(fit=True)
    picture = code.make_image(fill_color="black", back_color="white")
    picture = (picture.get_image() if hasattr(picture, "get_image") else picture).convert("RGB")
    side, m = max(96, img.width // 12), 14
    picture = picture.resize((side, side), Image.NEAREST)
    g = np.asarray(img.convert("L"))
    corners = [(m, m), (img.width - side - m, m), (m, img.height - side - m), (img.width - side - m, img.height - side - m)]
    free = [c for c in corners if not any(overlap((c[0], c[1], c[0] + side, c[1] + side), b) > 0 for b in boxes)] or corners
    best = min(free, key=lambda c: (g[c[1]:c[1] + side, c[0]:c[0] + side] < 128).mean())
    img.paste(picture, best)


def fill_template(page, spec, rng, side, qr_text=None, p_empty=P_EMPTY, p_hand=0.5, shuffle_keys=False):
    """A filled copy of a template page, its JSON answer and, for each key, how its value was written."""
    img, answer, kinds, boxes = page.copy(), {}, {}, {}
    fonts, crops = FORM_FONTS[side], CROPS[side]
    language = spec.get("language", "bilingual")
    fields, used = [f for f in spec["fields"] if f.get("key")], Counter()
    for f in fields:
        used[f["key"]] += 1                                    # a key marked on two boxes becomes key, key_2, ...
        key, box, style = f["key"] if used[f["key"]] == 1 else f"{f['key']}_{used[f['key']]}", f["box"], f.get("style", "box")
        kind = FIELD_KIND.get(f["key"], "free_text")
        script = "en" if (language == "en" or kind in ("name_en", "email")) else ("ar" if language == "ar" else rng.choice(["ar", "ar", "en"]))
        boxes[key] = box
        if rng.random() < p_empty:
            answer[key], kinds[key] = None, "empty"
            continue
        hand = rng.random() < p_hand
        if hand and kind in ("free_text", "address", "occupation", "company") and len(crops[script]) and rng.random() < 0.8:
            crop, text = crops[script].sample(rng)
            if paste_crop(img, box, crop, script == "ar", rng):
                answer[key], kinds[key] = text, "handwritten"
                continue
        value, shown = make_value(kind, rng, script, SENTENCES[script][1 if side == "heldout" else 0])
        arabic = arabic_font_needed(shown)
        if hand:
            font = rng.choice(fonts["ar_hand" if arabic else "en_hand"])
        else:
            font = rng.choice(fonts["mono"] if (not arabic and kind in ("reference", "invoice", "licence", "emirates_id", "passport") and rng.random() < 0.4)
                              else fonts["ar" if arabic else "en"])
        draw_value(img, box, shown, font, hand, rng, style)
        answer[key], kinds[key] = value, "handwritten" if hand else "printed"
    keys = list(answer)
    if shuffle_keys:
        rng.shuffle(keys)
    if qr_text:
        add_qr(img, qr_text, list(boxes.values()))
    return img, {k: answer[k] for k in keys}, {k: kinds[k] for k in keys}, {k: boxes[k] for k in keys}


def augraphy_pipeline():
    """Print, scan and fax damage without anything that adds marks or text (no scribbles, no markup), so labels stay exact."""
    import matplotlib.font_manager                     # Augraphy expects it to be imported
    from augraphy import (AugraphyPipeline, InkBleed, LowInkRandomLines, LowInkPeriodicLines, NoiseTexturize, BrightnessTexturize,
                          ColorPaper, DirtyRollers, LightingGradient, BadPhotoCopy, Faxify, Folding, ShadowCast, Jpeg, SubtleNoise)
    return AugraphyPipeline(ink_phase=[InkBleed(p=0.4), LowInkRandomLines(p=0.3), LowInkPeriodicLines(p=0.2)],
                            paper_phase=[ColorPaper(p=0.3), NoiseTexturize(p=0.5), BrightnessTexturize(p=0.5)],
                            post_phase=[DirtyRollers(p=0.2), LightingGradient(p=0.4), BadPhotoCopy(p=0.25), Faxify(p=0.15),
                                        Folding(p=0.3), ShadowCast(p=0.3), Jpeg(p=0.5), SubtleNoise(p=0.3)])


start = step("7c. held-out form set")
heldout_ready = [t for t in (split["heldout"] if split else []) if t in TEMPLATES and TEMPLATES[t]["keyed"]]
labels_path = os.path.join(HELDOUT_DIR, "labels.jsonl")
if not heldout_ready:
    print("   no held-out templates with keys marked yet (steps 1 and 2 above)")
elif saved_dataset("forms_heldout_digital") is not None and json.load(open(os.path.join(HELDOUT_DIR, "templates.json"))) == heldout_ready:
    print(f"   already built for {len(heldout_ready)} templates")
else:
    for sub in ["digital", "print"]:
        os.makedirs(os.path.join(HELDOUT_DIR, sub), exist_ok=True)
    writer, records = ShardWriter("forms_heldout_digital", FORM_FEATURES), []
    for t_index, tid in enumerate(heldout_ready):
        v = TEMPLATES[tid]
        page = rasterize(v["path"], v["spec"].get("page", 0))
        sheets = []
        for k in range(FILLS_PER_HELDOUT):
            form_id = f"AVF-{t_index + 1:02d}-{k + 1:03d}"
            rng = random.Random(f"{tid}-{k}")
            img, answer, kinds, boxes = fill_template(page, v["spec"], rng, "heldout", qr_text=form_id)
            img.save(os.path.join(HELDOUT_DIR, "digital", f"{form_id}.png"))
            sheets.append(img)
            record = {"id": form_id, "template": tid, "json": answer, "kinds": kinds, "boxes": boxes}
            records.append(record)
            writer.add({"image": image_bytes(img), "json": json.dumps(answer, ensure_ascii=False), "kinds": json.dumps(kinds),
                        "boxes": json.dumps(boxes), "template": tid, "id": form_id, "source": "heldout_digital"})
        sheets[0].save(os.path.join(HELDOUT_DIR, "print", f"{t_index + 1:02d}_{tid}.pdf"), save_all=True,
                       append_images=sheets[1:], resolution=150)       # 1,240 px at 150 dpi = A4 width
    with open(labels_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    ds = save(writer.finish(), "forms_heldout_digital")
    json.dump(heldout_ready, open(os.path.join(HELDOUT_DIR, "templates.json"), "w"))
    kinds_count = Counter(k for r in records for k in r["kinds"].values())
    print(f"   {len(ds)} held-out forms from {len(heldout_ready)} templates; values: {dict(kinds_count)}")
    print(f"   print these: {os.path.join(HELDOUT_DIR, 'print')} (one PDF per template, A4)")
    try:
        pipeline, writer = augraphy_pipeline(), ShardWriter("forms_heldout_augraphy", FORM_FEATURES)
        for r in records:
            base = np.asarray(Image.open(os.path.join(HELDOUT_DIR, "digital", f"{r['id']}.png")).convert("RGB"))[:, :, ::-1]
            for c in range(AUGRAPHY_COPIES):
                random.seed(f"{r['id']}-{c}")
                np.random.seed(int(hashlib.md5(f"{r['id']}-{c}".encode()).hexdigest()[:8], 16))
                out = pipeline(base.copy())
                out = out if out.ndim == 3 else np.stack([out] * 3, -1)
                writer.add({"image": image_bytes(Image.fromarray(np.ascontiguousarray(out[:, :, ::-1])), "JPEG", 90),
                            "json": json.dumps(r["json"], ensure_ascii=False), "kinds": json.dumps(r["kinds"]),
                            "boxes": json.dumps(r["boxes"]), "template": r["template"], "id": r["id"], "source": "heldout_augraphy"})
        ds = save(writer.finish(), "forms_heldout_augraphy")
        print(f"   {len(ds)} Augraphy copies (print, scan and fax damage)")
    except Exception as e:
        print(f"   Augraphy copies skipped ({type(e).__name__}: {str(e)[:120]})")
note("forms_heldout", templates=len(heldout_ready), fills_per_template=FILLS_PER_HELDOUT,
     fonts=FORM_FONTS["heldout"], handwriting="test splits only (KHATT official test or KITAB KHATT/OnlineKhatt; IAM and GNHK test)")

### 7d. Phone photos of the printed held-out forms
Put the photos in `MyDrive/arabic_vlm/form_photos/` (any names, JPEG or PNG, several photos of one sheet are fine).
Each photo's QR code is read (at three scales, with two detectors), matched to its label, and the photo is saved at
up to 2,000 px on the long side. Photos whose code cannot be read are listed; retake them with the code in view.

In [ ]:
def read_qr(img):
    arr = np.asarray(img.convert("RGB"))[:, :, ::-1]
    detectors = [cv2.QRCodeDetector()] + ([cv2.QRCodeDetectorAruco()] if hasattr(cv2, "QRCodeDetectorAruco") else [])
    for scale in (1.0, 0.5, 0.25):
        small = arr if scale == 1.0 else cv2.resize(arr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        for detector in detectors:
            try:
                text, _, _ = detector.detectAndDecode(small)
            except cv2.error:
                continue
            if text:
                return text.strip()
    return None


start = step("7d. form photos")
photos = sorted(p for p in glob.glob(os.path.join(FORM_PHOTOS_DIR, "**", "*"), recursive=True)
                if p.lower().endswith((".jpg", ".jpeg", ".png", ".heic", ".webp"))) if os.path.isdir(FORM_PHOTOS_DIR) else []
if photos and os.path.exists(labels_path):
    labels = {json.loads(l)["id"]: json.loads(l) for l in open(labels_path, encoding="utf-8")}
    writer, unread, unknown = ShardWriter("forms_heldout_photos", FORM_FEATURES), [], []
    for p in photos:
        try:
            img = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
        except Exception:
            unread.append(os.path.basename(p))
            continue
        code = read_qr(img)
        if code is None:
            unread.append(os.path.basename(p))
            continue
        if code not in labels:
            unknown.append(f"{os.path.basename(p)} ({code})")
            continue
        r = labels[code]
        scale = min(1.0, 2000 / max(img.size))
        if scale < 1.0:
            img = img.resize((int(img.width * scale), int(img.height * scale)), Image.LANCZOS)
        writer.add({"image": image_bytes(img, "JPEG", 90), "json": json.dumps(r["json"], ensure_ascii=False), "kinds": json.dumps(r["kinds"]),
                    "boxes": "{}", "template": r["template"], "id": r["id"], "source": "heldout_photo"})
    ds = save(writer.finish(), "forms_heldout_photos")
    print(f"   {len(ds)} photos matched to {len(set(ds['id']))} forms; {len(unread)} unreadable, {len(unknown)} with an unknown code")
    for name in (unread + unknown)[:20]:
        print("      retake:", name)
    note("forms_photos", photos=len(ds), forms=len(set(ds["id"])), unreadable=len(unread))
else:
    print("   no photos yet" if os.path.exists(labels_path) else "   no held-out forms yet (section 7c)")

### 7e. Training fills and the form entries of the test suite
The training templates are filled `N_TRAIN_FILLS` times with training fonts and training writers' handwriting, keys
in a random order half the time. 2% more fills (other seeds) are kept for the progress checks. Then the form sets
are added to the test suite. The photo set is the one entry that may grow later, as you add photos.

In [ ]:
start = step("7e. training fills")
train_ready = [t for t in (split["train"] if split else []) if t in TEMPLATES and TEMPLATES[t]["keyed"]]
if train_ready and (saved_dataset("forms_template_train") is None or manifest["parts"].get("forms_template_train", {}).get("templates") != train_ready):
    pages = {t: rasterize(TEMPLATES[t]["path"], TEMPLATES[t]["spec"].get("page", 0)) for t in train_ready}
    for name, count, offset in [("forms_template_train", N_TRAIN_FILLS, 0), ("val_forms_template", max(2, N_TRAIN_FILLS // 50), 10_000_000)]:
        writer = ShardWriter(name, FORM_FEATURES)
        for k in range(count):
            tid = train_ready[k % len(train_ready)]
            rng = random.Random(offset + k)
            img, answer, kinds, boxes = fill_template(pages[tid], TEMPLATES[tid]["spec"], rng, "train", p_empty=0.08,
                                                      shuffle_keys=rng.random() < 0.5)
            writer.add({"image": image_bytes(img, "JPEG", 90), "json": json.dumps(answer, ensure_ascii=False), "kinds": json.dumps(kinds),
                        "boxes": json.dumps(boxes), "template": tid, "id": f"T{k:05d}", "source": "template_train"})
        ds = save(writer.finish(), name)
        print(f"   {name}: {len(ds):,} fills of {len(train_ready)} training templates")
    note("forms_template_train", templates=train_ready, fills=N_TRAIN_FILLS)
else:
    print("   up to date" if train_ready else "   no training templates with keys yet")

FORM_SPEC = [("Held-out forms (digital)", "forms_heldout_digital", ["held-out templates", "held-out writers and fonts"]),
             ("Held-out forms (Augraphy)", "forms_heldout_augraphy", ["print, scan and fax damage"]),
             ("Held-out forms (photos)", "forms_heldout_photos", ["phone photos of prints", "grows as photos are added"]),
             ("FUNSD test", "funsd_test", ["real scanned English forms"])]
for name, dataset, tags in FORM_SPEC:
    ds = saved_dataset(dataset)
    if ds is None or len(ds) == 0:
        continue
    existing = next((e for e in suite["sets"] if e["name"] == name), None)
    if existing is not None and (dataset != "forms_heldout_photos" or existing["rows"] == len(ds)):
        continue
    entry = {"name": name, "dataset": dataset, "rows": len(ds), "indices": list(range(len(ds))), "headline": list(range(len(ds))),
             "family": None, "question": "forms", "kind": "form", "normaliser": None, "tags": tags, "created": time.strftime("%Y-%m-%d %H:%M")}
    suite["sets"] = [e for e in suite["sets"] if e["name"] != name] + [entry]
    print(f"   suite: {'updated' if existing else 'added'} {name} ({len(ds)} forms)")
json.dump(suite, open(SUITE_PATH, "w"), indent=0, ensure_ascii=False)

## 8. Unlabelled lines for self-labelling (optional)
Phase 2's self-labelling step reads unlabelled handwriting with two models and keeps a line only when both agree.
Two sources: line images you put in `MyDrive/arabic_vlm/unlabelled_lines/` (any folder structure; crops of single
lines), and the memoir blind-test lines, which come without transcriptions. Nothing here has a label yet, and none
of it is a test set: the memoir test split used for scoring is a different set of lines.

In [ ]:
start = step("8. unlabelled lines")
files = sorted(p for p in glob.glob(os.path.join(UNLABELLED_DIR, "**", "*"), recursive=True)
               if p.lower().endswith(IMAGE_EXTS)) if os.path.isdir(UNLABELLED_DIR) else []
if files and (saved_dataset("unlabelled_lines") is None or len(saved_dataset("unlabelled_lines")) != len(files)):
    writer = ShardWriter("unlabelled_lines", LINE_FEATURES)
    for p in files:
        try:
            with Image.open(p) as img:
                writer.add(line_row(img.convert("RGB"), "", "unlabelled", "", os.path.dirname(os.path.relpath(p, UNLABELLED_DIR)),
                                    os.path.basename(p), fmt="JPEG"))
        except Exception:
            pass
    ds = save(writer.finish(), "unlabelled_lines")
    print(f"   unlabelled_lines: {len(ds):,} images")
blind = saved_dataset("memoir_blind")
print(f"   your unlabelled lines: {len(files):,}; memoir blind-test lines: {len(blind) if blind is not None else 0:,}"
      + (" (they carry text, so they are not used as unlabelled)" if blind is not None and any(blind.select(range(min(50, len(blind))))["text"]) else ""))
note("unlabelled", user_lines=len(files), memoir_blind=len(blind) if blind is not None else 0)

## 9. Your own documents (optional, evaluation only)
As in v5: anonymised images in `MyDrive/arabic_vlm/real_docs/` with a `labels.csv` that has a `file` column and
either a `text` column (lines separated by `|`) or a `json` column (the fields as a JSON object). Copied into the
data folder (as image bytes, so the folder works anywhere) and scored by the training notebook, never trained on.

In [ ]:
real_path = os.path.join(REAL_DOCS_DIR, "labels.csv")
if os.path.exists(real_path):
    writer = ShardWriter("real_docs", Features({"image": ImageFeature(), "text": Value("string"), "json": Value("string"), "file": Value("string")}))
    for r in csv.DictReader(open(real_path, encoding="utf-8")):
        file = os.path.join(REAL_DOCS_DIR, r["file"])
        if not os.path.exists(file):
            print(f"   missing image: {file}")
            continue
        with Image.open(file) as img:
            writer.add({"image": image_bytes(ImageOps.exif_transpose(img).convert("RGB"), "JPEG", 92),
                        "text": (r.get("text") or "").replace("|", "\n").strip(), "json": (r.get("json") or "").strip(), "file": r["file"]})
    ds = save(writer.finish(), "real_docs")
    note("real_docs", images=len(ds), use="evaluation only")
    print(f"   real documents: {len(ds)} images saved")
else:
    print(f"no real documents found at {real_path} (optional)")

## 10. Manifest and summary
`manifest.json` records what every step produced and each source's terms. The model is for research use only, so
every source can be used under its research terms; the licence column matters only if that ever changes.

To save Drive space once everything is built, set `DELETE_RAW = True` below: it removes `raw/` (the unfiltered
training downloads). Only do that when you will not add test sets later, because a new test set means filtering the
training sources again.

In [ ]:
DELETE_RAW = False
manifest["size_gb"] = round(sum(os.path.getsize(os.path.join(root, f)) for root, _, fs in os.walk(DATA_DIR) for f in fs) / 2**30, 2)
manifest["datasets"] = {name: len(load_from_disk(dataset_path(name)))
                        for name in sorted(os.listdir(os.path.join(DATA_DIR, "datasets")))
                        if os.path.exists(os.path.join(dataset_path(name), "state.json"))}
json.dump(manifest, open(MANIFEST_PATH, "w"), indent=1, ensure_ascii=False)
print(f"data_v6: {manifest['size_gb']} GB in {DATA_DIR}\n")
print(f"{'dataset':34s}{'rows':>9s}")
for name, n in manifest["datasets"].items():
    print(f"{name:34s}{n:9,d}")
if DELETE_RAW:
    shutil.rmtree(os.path.join(DATA_DIR, "raw"), ignore_errors=True)
    print("raw/ deleted")
shutil.rmtree(os.path.join(WORK_DIR, "shards"), ignore_errors=True)
print("\nNext: the v6 training notebook (part 2 v6) reads this folder. Add the official KHATT release, templates or photos "
      "at any time and run this notebook again; only the new parts are built.")